## Initial BLASTp Search

In [5]:
import os
import time
import subprocess
import pandas as pd
from Bio import SeqIO

In [ ]:
pep_folder = 'data/1k-species-data/y1000p_pep_files'
db_folder = 'data/1k-species-data/blast_dbs'
hits_folder = 'data/1k-species-data/CHC1_hits'
master_fasta = 'data/1k-species-data/1k-CHC1.fasta'
metadata_csv = 'data/1k-species-data/CHC1_metadata.csv'
query_fasta = 'data/1k-species-data/CHC1_seed.fasta'

'''
Add fuzzy-string logic (fuzzy wuzzy, other tool) to account for human error.
'''

metadata = []
species_found = []

for pep_file in os.listdir(pep_folder):
    if not pep_file.endswith('.final.pep'):
        continue
    
    if pep_file.startswith('yH'):
        species_name = pep_file.split('_', 1)[1].rsplit('_', 1)[0]
        for other_file in os.listdir(pep_folder):
            if species_name in other_file and 'final' in other_file and not other_file.startswith('yH'):
                pep_file = other_file
    else:
        species_name = pep_file.rsplit('.', 2)[0]
                
    print(pep_file)
    print(species_name)

    if species_name in species_found:
        continue
    species_found.append(species_name)
    print(f'Processing {species_name}')
    pep_path = os.path.join(pep_folder, pep_file)

    species_folder = os.path.join(db_folder, species_name)
    os.makedirs(species_folder, exist_ok=True)
    db_path = os.path.join(species_folder, species_name)
    subprocess.run([
        'makeblastdb',
        '-in', pep_path,
        '-dbtype', 'prot',
        '-out', db_path
    ], check=True)
    
    hits_file = os.path.join(hits_folder, f"{species_name}_CHC1_hits.txt")
    subprocess.run([
        'blastp',
        '-query', query_fasta,
        '-db', db_path,
        '-out', hits_file,
        '-outfmt', '6'
    ], check=True)

    time.sleep(0.1)
    
    if os.path.exists(hits_file):
        with open(hits_file) as f:
            lines = f.readlines()
        if lines:
            # BLAST outfmt 6: qseqid sseqid pident length mismatch gapopen qstart qend sstart send evalue bitscore
            top_hit = max(lines, key=lambda x: float(x.split()[2]))  # highest % identity
            cols = top_hit.strip().split()
            sseqid, pident, evalue, bitscore = cols[1], float(cols[2]), float(cols[10]), float(cols[11])
            
            seq_lines = []
            record = False
            with open(pep_path) as pep_f:
                for line in pep_f:
                    if line.startswith('>'):
                        record = sseqid in line
                        continue
                    if record:
                        seq_lines.append(line.strip())
            seq = '\n'.join([line.strip().rstrip('*') for line in seq_lines])
            
            with open(master_fasta, 'a') as f_out:
                f_out.write(f'>{species_name}|{sseqid}\n{seq}\n')
            
            metadata.append([species_name, sseqid, len(seq.replace('\n','')), pident, evalue, bitscore])
        else:
            print(f'No hits found for {species_name}')

df = pd.DataFrame(metadata, columns=['Species', 'Hit_ID', 'Seq_Length', 'Percent_Identity', 'Evalue', 'Bitscore'])
df.to_csv(metadata_csv, index=False)

## BLAST Search with Saccharomyces Cerevisiae CHC1 over Orthogroup FASTAs

In [1]:
local_path = '/Users/georgecrawford/Documents/yeast-clathrin-conservation/'

In [2]:
import os
from pathlib import Path

orthogroup_dir = Path(local_path + 'genome_analyses/y1000p_orthofinder/Orthogroup_Sequences').expanduser()
os.chdir(orthogroup_dir)
print('Current folder:', os.getcwd())

Current folder: /Users/georgecrawford/Documents/yeast-clathrin-conservation/genome_analyses/y1000p_orthofinder/Orthogroup_Sequences


In [4]:
all_fasta_file = 'all_orthogroups.fa'

fasta_files = list(orthogroup_dir.glob('*.fa')) + list(orthogroup_dir.glob('*.fasta'))
print(f"Found {len(fasta_files)} FASTA files.")

with open(all_fasta_file, 'w') as outfile:
    for fasta in fasta_files:
        with open(fasta, 'r') as infile:
            outfile.write(infile.read())

print(f"All orthogroups written to {all_fasta_file}")

Found 72382 FASTA files.
All orthogroups written to all_orthogroups.fa


In [73]:
!makeblastdb -in all_orthogroups.fa -dbtype prot



Building a new DB, current time: 11/04/2025 15:01:50
New DB name:   /Users/georgecrawford/Documents/yeast-clathrin-conservation/genome_analyses/y1000p_orthofinder/Orthogroup_Sequences/all_orthogroups.fa
New DB title:  all_orthogroups.fa
Sequence type: Protein
Keep MBits: T
Maximum file size: 3000000000B
Adding sequences from FASTA; added 6971672 sequences in 58.8001 seconds.




In [76]:
ss_chc1_file = local_path + "genome_analyses/blast_results/s_cerevisiae_CHC1.fasta"
blast_output_file = local_path + "genome_analyses/blast_results/SS_CHC1_vs_orthogroups.txt"

!blastp -query {ss_chc1_file} -db all_orthogroups.fa -out {blast_output_file} -outfmt 6 -max_target_seqs 5

In [78]:
with open(blast_output_file) as f:
    lines = f.readlines()
    print('Top BLAST hits:')
    for line in lines[:10]:
        print(line.strip())

Top BLAST hits:
saccharomyces_cerevisiae_YGL206C	YGL206C|saccharomyces_cerevisiae.sgd	100.000	1653	0	0	1	1653	1	1653	0.0	3387
saccharomyces_cerevisiae_YGL206C	g004661.m1|saccharomyces_cerevisiae.final	100.000	1653	0	0	1	1653	1	1653	0.0	3387
saccharomyces_cerevisiae_YGL206C	g004575.m1|saccharomyces_paradoxus.final	98.246	1653	29	0	1	1653	1	1653	0.0	3336
saccharomyces_cerevisiae_YGL206C	g004716.m1|saccharomyces_mikatae.final	96.854	1653	52	0	1	1653	1	1653	0.0	3300
saccharomyces_cerevisiae_YGL206C	g004535.m1|saccharomyces_arboricola.final	96.310	1653	61	0	1	1653	1	1653	0.0	3281


### Top sequence's ID (YGL206C) found in orthogroup with ID 0000944 

#### Next steps:
- Determine actually how many species are in this orthogroup (total number of sequences in it is 1307, but this is because many of them have two annotations -- one internal (with the format g######.m1) and one universal (like Saccharomyces cerevisiae's YGL206C). The total number needs to be less anyway (1154).
- Run a reciprocal BLAST of orthogroup's sequences to S. cerevisiae's proteome. This will fully confirm that these are CHC1s.
- Deal with the multiple abnormally short sequences present in this orthogroup FASTA file. The output below shows the sequence lengths.

In [96]:
CHC_OG_records = list(SeqIO.parse(local_path + 'genome_analyses/y1000p_orthofinder/Orthogroup_Sequences/OG0000944.fasta', 'fasta'))

print([len(seq_record.seq) for seq_record in CHC_OG_records])

[1681, 1680, 1686, 1675, 1665, 1702, 1690, 1678, 1666, 1704, 1676, 1685, 1682, 1682, 1674, 1675, 1649, 1676, 1681, 1670, 1660, 1656, 1647, 1665, 1671, 1643, 1635, 1635, 1669, 1673, 1673, 1643, 865, 727, 1653, 1651, 1649, 1651, 1683, 1674, 1653, 1643, 1664, 1653, 1654, 1654, 1654, 1656, 1656, 1658, 1657, 1673, 1658, 1665, 1665, 1665, 1665, 1665, 1665, 1662, 1660, 1678, 1664, 1526, 121, 1653, 1653, 1653, 1652, 1631, 1653, 1653, 1653, 1653, 1670, 1673, 1673, 1672, 2889, 1668, 1668, 1666, 1681, 1672, 1666, 1673, 330, 399, 789, 1673, 1673, 1604, 1662, 1669, 1655, 1655, 1655, 283, 95, 1472, 512, 1655, 1653, 1655, 1653, 1653, 1653, 1654, 1655, 1655, 1654, 1655, 1650, 1637, 1629, 1637, 1650, 1654, 1655, 995, 652, 1654, 1654, 1655, 1655, 1654, 1653, 1654, 1626, 1655, 182, 1655, 1659, 1648, 1654, 1652, 1664, 1654, 1648, 1654, 1651, 1652, 1661, 1670, 1659, 1720, 1667, 1658, 1653, 166, 1656, 1657, 1653, 1652, 1663, 1667, 1668, 1670, 1648, 1671, 1657, 1659, 1658, 1670, 1654, 1653, 1661, 1665, 1662,

### S. Cerevisiae CLC1 (YGR167W) found in orthogroup with ID 0002385

Below are the lengths of the sequences in this orthogroup (need to do further analysis, but from first glance it definitely seems like most/all are full sequences)

There are also a total of 1208 sequences in this orthogroup, which is more than the needed 1154. Again, this is likely due to certain sequences being repeated due to separate annotations. Regardless, next steps include:
- Finding how many unique sequences there are in this orthogroup
- Run a reciprocal BLAST of orthogroup's sequences to S. cerevisiae's proteome. This will fully confirm that these are CHC1s.

In [100]:
CLC_OG_records = list(SeqIO.parse(local_path + 'genome_analyses/y1000p_orthofinder/Orthogroup_Sequences/OG0002385.fasta', 'fasta'))

print([len(seq_record.seq) for seq_record in CLC_OG_records])

[253, 242, 290, 319, 281, 301, 277, 266, 196, 260, 236, 246, 291, 221, 241, 228, 157, 240, 233, 233, 225, 225, 236, 218, 225, 217, 222, 222, 222, 222, 229, 232, 209, 149, 155, 157, 146, 239, 236, 226, 215, 249, 201, 235, 229, 241, 217, 217, 217, 216, 238, 213, 212, 213, 213, 213, 213, 213, 204, 225, 244, 249, 134, 236, 233, 233, 234, 236, 236, 236, 234, 234, 230, 238, 239, 237, 212, 217, 229, 232, 218, 228, 217, 218, 218, 229, 253, 221, 270, 235, 238, 238, 235, 236, 201, 217, 204, 205, 216, 219, 216, 194, 199, 217, 220, 200, 216, 215, 223, 211, 212, 201, 192, 120, 216, 217, 229, 220, 212, 199, 216, 229, 221, 760, 238, 125, 203, 212, 217, 229, 125, 215, 208, 156, 228, 230, 201, 215, 212, 210, 218, 218, 240, 177, 200, 220, 212, 197, 125, 218, 222, 226, 221, 217, 188, 230, 119, 218, 227, 220, 222, 213, 209, 199, 215, 243, 222, 227, 202, 216, 202, 230, 213, 211, 175, 219, 207, 217, 206, 220, 213, 235, 209, 234, 236, 192, 204, 202, 202, 213, 221, 218, 212, 236, 221, 210, 224, 242, 230, 226,

In [104]:
len(CHC_OG_records)

1307

In [146]:
duplicate_species_CHC = []

for i in range(len(CHC_OG_records)):
    species_name = CHC_OG_records[i].id.split('|')[1].split('.')[0]
    if species_name.startswith('yH'):
        species_name = species_name.split('_', 1)[1]
        
    for j in range(len(CHC_OG_records)):
        if i == j:
            continue
            
        other_species_name = CHC_OG_records[j].id.split('|')[1].split('.')[0]
        if other_species_name.startswith('yH'):
            other_species_name = other_species_name.split('_', 1)[1]
        
        if species_name.lower() == other_species_name.lower() and species_name.lower() not in duplicate_species_CHC:
            duplicate_species_CHC.append(species_name.lower())
            print(f'Duplicate sequences found for {species_name}')

Duplicate sequences found for Postia_placenta
Duplicate sequences found for candida_albicans
Duplicate sequences found for clavispora_lusitaniae
Duplicate sequences found for pichia_kudriavzevii
Duplicate sequences found for saccharomyces_cerevisiae
Duplicate sequences found for saprochaete_suaveolens
Duplicate sequences found for spathaspora_gorwiae
Duplicate sequences found for kazachstania_slooffiae_160519
Duplicate sequences found for kazachstania_exigua_160519
Duplicate sequences found for kazachstania_spencerorum_160519
Duplicate sequences found for kazachstania_yakushimaensis_160519
Duplicate sequences found for citeromyces_nyonsensis_180604
Duplicate sequences found for Yamadazyma_sp_nov_plate33_SPADES
Duplicate sequences found for Vanderwaltozyma_sp_nov_plate33_SPADES
Duplicate sequences found for Metschnikowia_sp_nov_plate33_SPADES
Duplicate sequences found for Suhomyces_sp_nov_plate33_SPADES
Duplicate sequences found for geotrichum_klebahnii_190924
Duplicate sequences found 

In [147]:
len(duplicate_species_CHC)

87

### For Quick MSA

In [179]:
from Bio import SeqIO
import subprocess 
from io import StringIO 
from Bio.SeqRecord import SeqRecord

duplicates = list(SeqIO.parse(local_path + 'genome_analyses/dupes.fasta', 'fasta'))
output = StringIO()
SeqIO.write(duplicates, output, 'fasta')
duplicates_str = output.getvalue()

child = subprocess.Popen(    
    ['mafft', '--localpair', '--maxiterate', '1000', '--ep', '0', '--quiet', '-'], 
    stdin=subprocess.PIPE,    
    stdout=subprocess.PIPE,   
    stderr=subprocess.PIPE      
)

child_out, child_err = child.communicate(input=duplicates_str.encode())

aligned_duplicates = list(SeqIO.parse(StringIO(child_out.decode()), 'fasta'))

In [180]:
with open(local_path + 'genome_analyses/aligned_dupes.fasta', 'w') as output_handle:
    SeqIO.write(aligned_duplicates, output_handle, 'fasta')

### Check Interface Residues (old version; doesn't account for '-' symbols)

In [227]:
K1326 = 'AILYSKFKPQ'
K1415 = 'YLEFKPLLLN'
W113 = 'RKWREEQMER'
W135 = 'QEAEWKEKAI'
W146 = 'KELEEWYARQ'

aligned_dupes = list(SeqIO.parse(local_path + 'genome_analyses/aligned_dupes.fasta', 'fasta'))

for seq in aligned_dupes:
    print(seq.description)

print('')

chain_type = input('Enter CHC1 or CLC1: ')

human_seq = None
for seq in aligned_dupes:
    if 'human' in seq.description.lower():
        human_seq = str(seq.seq)
        break

if human_seq is None:
    raise ValueError('Please include the human CHC17 sequence in the dupes.fasta file and rerun the cells above.')

yeast_residues_dict = {}

residue_list = []
if chain_type == 'CHC1':
    residue_list = [K1326, K1415]
elif chain_type == 'CLC1':
    residue_list = [W113, W135, W146]
else:
    raise ValueError('Please enter "CHC1" or "CLC1".')

for motif in residue_list:
    start = human_seq.find(motif)

    if start != -1 and chain_type == 'CHC1':
        k_index_in_motif = motif.index('K')
        k_position = start + k_index_in_motif

        print(f'K at 0-based index {k_position}')

        residues_for_motif = []
        for seq in aligned_dupes:
            if 'human' not in seq.description.lower():
                yeast_seq = str(seq.seq)
                interface_residue = yeast_seq[k_position]
                residues_for_motif.append(interface_residue)
                print(interface_residue)
        yeast_residues_dict[motif] = residues_for_motif    
        
    elif start != -1 and chain_type == 'CLC1':
        w_index_in_motif = motif.index('W')
        w_position = start + w_index_in_motif

        print(f'W at 0-based index {w_position}')

        residues_for_motif = []
        for seq in aligned_dupes:
            if 'human' not in seq.description.lower():
                yeast_seq = str(seq.seq)
                interface_residue = yeast_seq[w_position]
                residues_for_motif.append(interface_residue)
                print(interface_residue)
        yeast_residues_dict[motif] = residues_for_motif    
    
    else:
        print('Motif not found')

for motif, residues in yeast_residues_dict.items():
    if motif == 'AILYSKFKPQ':
        motif_name = 'K1326'
    elif motif == 'YLEFKPLLLN':
        motif_name = 'K1415'
    elif motif == 'RKWREEQMER':
        motif_name = 'W113'
    elif motif == 'QEAEWKEKAI':
        motif_name = 'W135'
    elif motif == 'KELEEWYARQ':
        motif_name = 'W146'
        
    if len(set(residues)) == 1:
        print(f'All yeast residues for motif {motif_name} are the same: {residues[0]}')
    else:
        print(f'Variation in yeast residues for motif {motif_name}: {residues}')

sp|P09496-2|CLCA_HUMAN Isoform Non-brain of Clathrin light chain A OS=Homo sapiens OX=9606 GN=CLTA
g003999.m1|yHMPu5000034581_schwanniomyces_occidentalis_var_occidentalis_170713.final
g001913.m1|yHMPu5000041781_schwanniomyces_occidentalis_var_persoonii_190924.final



Enter CHC1 or CLC1:  CLC1


Motif not found
W at 0-based index 143
K
K
W at 0-based index 154
N
N
All yeast residues for motif W135 are the same: K
All yeast residues for motif W146 are the same: N


### Check Interface Residues (new version to account for '-' symbols, but CONFIRM THIS CODE)

In [249]:
def ungapped_to_gapped_index(gapped_seq: str, ungapped_index: int) -> int:
    """
    Convert an index in the ungapped sequence to the corresponding index
    (column) in the gapped alignment sequence.
    """
    count = 0
    for i, aa in enumerate(gapped_seq):
        if aa != '-':
            if count == ungapped_index:
                return i
            count += 1
    raise ValueError("Ungapped index is out of range for this gapped sequence.")


K1326 = 'AILYSKFKPQ'
K1415 = 'YLEFKPLLLN'
W113  = 'RKWREEQMER'
W135  = 'QEAEWKEKAI'
W146  = 'KELEEWYARQ'

aligned_dupes = list(SeqIO.parse(local_path + 'genome_analyses/aligned_dupes.fasta', 'fasta'))

for seq in aligned_dupes:
    print(seq.description)
print('')

chain_type = input('Enter CHC1 or CLC1: ').strip().upper()

if chain_type == 'CHC1':
    residue_list = [K1326, K1415]
    anchor_aa = 'K'
elif chain_type == 'CLC1':
    residue_list = [W113, W135, W146]
    anchor_aa = 'W'
else:
    raise ValueError('Please enter "CHC1" or "CLC1".')

human_seq = None
for seq in aligned_dupes:
    d = seq.description.lower()
    if 'human' not in d:
        continue

    # For CLC1, prefer human CLTA / light chain A
    if chain_type == 'CLC1' and ('clta' in d or 'light chain' in d or 'clca_human' in d):
        human_seq = str(seq.seq).upper()
        break

    # For CHC1, prefer human clathrin heavy chain (CLTC/CHC17)
    if chain_type == 'CHC1' and ('cltc' in d or 'chc17' in d or 'heavy chain' in d or 'clathrin heavy' in d):
        human_seq = str(seq.seq).upper()
        break

if human_seq is None:
    for seq in aligned_dupes:
        if 'human' in seq.description.lower():
            human_seq = str(seq.seq).upper()
            break

if human_seq is None:
    raise ValueError('No human sequence found in aligned_dupes.fasta.')

human_aln = human_seq                      # gapped alignment sequence
human_ungapped = human_aln.replace('-', '')  # ungapped sequence for motif searching

yeast_residues_dict = {}

for motif in residue_list:
    start_ungapped = human_ungapped.find(motif)
    if start_ungapped == -1:
        print(f'Motif not found in human (ungapped): {motif}')
        continue

    # position in UNGAPPED human sequence of the key residue (K or W) within the motif
    offset = motif.index(anchor_aa)
    target_ungapped_pos = start_ungapped + offset

    # map to alignment column (0-based)
    alignment_col = ungapped_to_gapped_index(human_aln, target_ungapped_pos)
    print(f'{anchor_aa} for motif {motif} is at alignment column (0-based): {alignment_col}')

    residues_for_motif = []
    for seq in aligned_dupes:
        if 'human' in seq.description.lower():
            continue
        yeast_aln = str(seq.seq).upper()
        residue = yeast_aln[alignment_col] if alignment_col < len(yeast_aln) else None
        residues_for_motif.append(residue)

    yeast_residues_dict[motif] = residues_for_motif

motif_to_name = {
    K1326: 'K1326',
    K1415: 'K1415',
    W113:  'W113',
    W135:  'W135',
    W146:  'W146',
}

print('')
for motif, residues in yeast_residues_dict.items():
    motif_name = motif_to_name.get(motif, motif)

    cleaned = [r for r in residues if r not in (None, '-', 'X')]
    if not cleaned:
        print(f'No usable residues for motif {motif_name} (all gaps/unknown).')
        continue

    if len(set(cleaned)) == 1:
        print(f'All yeast residues for motif {motif_name} are the same: {cleaned[0]}')
    else:
        print(f'Variation in yeast residues for motif {motif_name}: {cleaned}')

sp|P09496-2|CLCA_HUMAN Isoform Non-brain of Clathrin light chain A OS=Homo sapiens OX=9606 GN=CLTA
g005004.m1|yHMPu5000038322_schwanniomyces_polymorphus_var_africanus_170713.final
g000631.m1|yHMPu5000041780_Schwanniomyces_polymorphus_var_polymorphus_SPADES.final



Enter CHC1 or CLC1:  CLC1


W for motif RKWREEQMER is at alignment column (0-based): 120
W for motif QEAEWKEKAI is at alignment column (0-based): 142
W for motif KELEEWYARQ is at alignment column (0-based): 153

All yeast residues for motif W113 are the same: Q
No usable residues for motif W135 (all gaps/unknown).
All yeast residues for motif W146 are the same: W


### CLC1 Data Cleaning

In [148]:
duplicate_species_CLC = []

for i in range(len(CLC_OG_records)):
    species_name = CLC_OG_records[i].id.split('|')[1].split('.')[0]
    if species_name.startswith('yH'):
        species_name = species_name.split('_', 1)[1]
        
    for j in range(len(CLC_OG_records)):
        if i == j:
            continue
            
        other_species_name = CLC_OG_records[j].id.split('|')[1].split('.')[0]
        if other_species_name.startswith('yH'):
            other_species_name = other_species_name.split('_', 1)[1]
        
        if species_name.lower() == other_species_name.lower() and species_name.lower() not in duplicate_species_CLC:
            duplicate_species_CLC.append(species_name.lower())
            print(f'Duplicate sequences found for {species_name}')

Duplicate sequences found for Postia_placenta
Duplicate sequences found for candida_albicans
Duplicate sequences found for candida_parapsilosis
Duplicate sequences found for saccharomyces_cerevisiae
Duplicate sequences found for kazachstania_jainica_160519
Duplicate sequences found for kazachstania_exigua_160519
Duplicate sequences found for kazachstania_spencerorum_160519
Duplicate sequences found for Yamadazyma_sp_nov_plate33_SPADES
Duplicate sequences found for Metschnikowia_sp_nov_plate33_SPADES
Duplicate sequences found for Suhomyces_sp_nov_plate33_SPADES
Duplicate sequences found for geotrichum_klebahnii_190924
Duplicate sequences found for candida_digboiensis_210210_SPADES
Duplicate sequences found for Candida_sp
Duplicate sequences found for Starmera_quercuum_p31_SPADES
Duplicate sequences found for pichia_sp_170307
Duplicate sequences found for Magnusiomyces_magnusii_S134_MASURCA
Duplicate sequences found for martiniozyma_abiesophila_170307
Duplicate sequences found for blasto

In [149]:
len(duplicate_species_CLC)

43

In [63]:
colorstrip_path = local_path + 'fig2_tree_data/itol_color_strip.txt'


with open(colorstrip_path) as f:
    for i, line in enumerate(f, start=1):
        if '\t' not in line and not line.startswith('#'):
            print(i, line.strip())

1 DATASET_COLORSTRIP
4 
5 SEPARATOR TAB
8 
13 
15 
20 
23 
27 
30 
31 
36 
40 DATA


In [64]:
with open(colorstrip_path, 'r') as f:
    lines = f.readlines()

#data_start = next(i for i, line in enumerate(lines) if line.strip() == 'DATA') + 1
#print(data_start)

data_rows = []
for line in lines[42:]:
    line = line.strip()
    if not line:
        continue
    parts = line.split()
    if len(parts) != 3:
        raise ValueError(f'Unexpected number of columns in line: {line}')
    data_rows.append(parts)


species_clade_df = pd.DataFrame(data_rows, columns=['species', 'fill_color', 'border_color'])

species_clade_df = species_clade_df.apply(lambda x: x.str.strip())

print(species_clade_df.shape)
species_clade_df

(1175, 3)


,species,fill_color,border_color
0,Nakazawaea_anatomiae,#6A3906,#6A3906
1,Nakazawaea_ernobii,#6A3906,#6A3906
2,Nakazawaea_holstii,#6A3906,#6A3906
3,Nakazawaea_ishiwadae,#6A3906,#6A3906
4,Nakazawaea_laoshanensis,#6A3906,#6A3906
...,...,...,...
1170,Tortispora_starmeri,#FF0090,#FF0090
1171,Trigonopsis_californica,#FF0090,#FF0090
1172,Trigonopsis_cantarellii,#FF0090,#FF0090
1173,Trigonopsis_variabilis,#FF0090,#FF0090


In [65]:
list(species_clade_df['fill_color']) == list(species_clade_df['border_color'])

True

In [66]:
species_clade_df = species_clade_df.drop(columns='border_color').rename(columns={'fill_color': 'clade_color'})
species_clade_df

,species,clade_color
0,Nakazawaea_anatomiae,#6A3906
1,Nakazawaea_ernobii,#6A3906
2,Nakazawaea_holstii,#6A3906
3,Nakazawaea_ishiwadae,#6A3906
4,Nakazawaea_laoshanensis,#6A3906
...,...,...
1170,Tortispora_starmeri,#FF0090
1171,Trigonopsis_californica,#FF0090
1172,Trigonopsis_cantarellii,#FF0090
1173,Trigonopsis_variabilis,#FF0090


In [67]:
duplicates_bool_list = species_clade_df['species'].duplicated(keep=False)
print('Number of unique species names:', len(duplicates_bool_list[duplicates_bool_list == False]))
duplicates_df = species_clade_df[duplicates_bool_list]
duplicates_df

Number of unique species names: 1175


,species,clade_color


In [68]:
species_clade_df[species_clade_df['species'].str.contains('wickerhamomyces_sp', case=False)]

,species,clade_color
382,Wickerhamomyces_sp._UFMG-CM-Y6624,#00e4ff
383,Wickerhamomyces_sp._NRRL_Y-7574,#00e4ff
384,Wickerhamomyces_sp._NRRL_YB-3031,#00e4ff
385,Wickerhamomyces_sp._yHMH26,#00e4ff
386,Wickerhamomyces_sp._yHMH451,#00e4ff
387,Wickerhamomyces_sp._yHMH617,#00e4ff
388,Wickerhamomyces_sp._yHQL14,#00e4ff
389,Wickerhamomyces_sp._NRRL_YB-2243,#00e4ff


In [69]:
for line in lines:
    if line.startswith('LEGEND_COLORS'):
        colors = line.strip().split()[1:]
    if line.startswith('LEGEND_LABELS'):
        labels = line.strip().split()[1:]

color_to_clade = dict(zip(colors, labels))

species_clade_df['clade_name'] = species_clade_df['clade_color'].map(color_to_clade)

In [70]:
species_clade_df

,species,clade_color,clade_name
0,Nakazawaea_anatomiae,#6A3906,Alaninetales
1,Nakazawaea_ernobii,#6A3906,Alaninetales
2,Nakazawaea_holstii,#6A3906,Alaninetales
3,Nakazawaea_ishiwadae,#6A3906,Alaninetales
4,Nakazawaea_laoshanensis,#6A3906,Alaninetales
...,...,...,...
1170,Tortispora_starmeri,#FF0090,Trigonopsidales
1171,Trigonopsis_californica,#FF0090,Trigonopsidales
1172,Trigonopsis_cantarellii,#FF0090,Trigonopsidales
1173,Trigonopsis_variabilis,#FF0090,Trigonopsidales


In [71]:
cleaned_CHCs = list(SeqIO.parse(local_path + 'genome_analyses/cleaned_CHC1_OG.fasta', 'fasta'))

matched_names = []
unmatched_names = []

for seq in cleaned_CHCs:
    if any(species.lower() in seq.id.lower() for species in species_clade_df['species']):
        matched_names.append(seq.id)
    else:
        unmatched_names.append(seq.id)

print(f'Number of matched names: {len(matched_names)}')
print(f'Number of unmatched names: {len(unmatched_names)}')

Number of matched names: 826
Number of unmatched names: 331


In [72]:
cleaned_CHCs = list(SeqIO.parse(local_path + 'genome_analyses/cleaned_CHC1_OG.fasta', 'fasta'))

matched_names = []
unmatched_names = []
duplicate_dict = {}

species_list = species_clade_df['species'].str.lower()

for seq in cleaned_CHCs: # Fix
    seq_id = seq.id.lower()
    found = False
    
    for species in species_list:
        if species in seq_id:
            matched_names.append(seq.id)
            duplicate_dict[species] = duplicate_dict.get(species, 0) + 1
            found = True
            break
            
    if not found:
        unmatched_names.append(seq.id)

duplicate_names = [species_name for species_name in duplicate_dict if duplicate_dict[species_name] > 1]

print(f'Number of matched names: {len(matched_names)}')
print(f'Number of duplicate names: {len(duplicate_names)}')
print(f'Number of unmatched names: {len(unmatched_names)}')

Number of matched names: 826
Number of duplicate names: 6
Number of unmatched names: 331


In [73]:
no_strain_suffixes = []

for species_name in list(species_clade_df['species']):
    name_parts = species_name.split('_')
    new_name = name_parts[0] + '_' + name_parts[1]
    no_strain_suffixes.append(new_name)

species_clade_df['species'] = no_strain_suffixes

In [74]:
species_clade_df = species_clade_df.drop_duplicates(subset=['species']).reset_index(drop=True)

In [75]:
len(species_clade_df['species'])

1096

In [76]:
matched_names = []
unmatched_names = []

for seq in cleaned_CHCs:
    if any(species.lower() in seq.id.lower() for species in species_clade_df['species']):
        matched_names.append(seq.id)
    else:
        unmatched_names.append(seq.id)

print(f'Number of matched names: {len(matched_names)}')
print(f'Number of unmatched names: {len(unmatched_names)}')

Number of matched names: 880
Number of unmatched names: 277


In [77]:
species_table = pd.read_csv(local_path + 'data/species_table.csv')
species_table.head()

,assembly_fullID_updated,Y1000+_ID,NEW_Tip_ID,Order,Species,Strain_Sequenced,NRRL,CBS,Other,CTH,...,Missing,Type strain,Type Species,Y1000+ Project,Genome_Acc,Reads_Acc,Citation Number Reference,Reference,Citation number for Related Pub,Related_Prior_Pub
0,yHMPu5000035005_lodderomyces_elongisporus_1605...,yHMPu5000035005,Lodderomyces_elongisporus,Serinales,Lodderomyces elongisporus,NRRL YB-4239,NRRL YB-4239,CBS 2605,NaN,NaN,...,1.12%,Yes,Yes,Yes,JANIWC000000000,SRR16974325,NaN,New Y1000+,134,"Butler G, et al. Evolution of pathogenicity an..."
1,yHMPu5000034659_debaryomyces_hansenii_180604,yHMPu5000034659,Debaryomyces_hansenii,Serinales,Debaryomyces hansenii,NRRL Y-7426,NRRL Y-7426,CBS 767,NaN,NaN,...,0.33%,Yes,Yes,Yes,JAKTQS000000000,SRR16974357,NaN,New Y1000+,140,"Dujon B, et al. Genome evolution in yeasts. Na..."
2,yHMPu5000034723_candida_glabrata_180604,yHMPu5000034723,Nakaseomyces_glabratus,Saccharomycetales,Nakaseomyces glabratus,NRRL Y-65,NRRL Y-65,CBS 138,NaN,NaN,...,0.47%,Yes,No,Yes,JAKTPX000000000,SRR16974273,NaN,New Y1000+,140,"Dujon B, et al. Genome evolution in yeasts. Na..."
3,yHMPu5000040963_lachancea_quebecuensis_200128,yHMPu5000040963,Lachancea_quebecensis,Saccharomycetales,Lachancea quebecensis,CBS 14138,NaN,CBS 14138,NaN,NaN,...,0.14%,Yes,No,Yes,JANIJM000000000,SRR16988846,NaN,New Y1000+,142,"Freel KC, et al. Whole-Genome Sequencing and I..."
4,yHMPu5000035666_nakaseomyces_bacillisporus_160613,yHMPu5000035666,Nakaseomyces_bacillisporus,Saccharomycetales,Nakaseomyces bacillisporus,NRRL Y-17846,NRRL Y-17846,CBS 7720,NaN,NaN,...,0.66%,Yes,No,Yes,JAJMDR000000000,SRR16988934,NaN,New Y1000+,143,"Gabaldon T, et al. Comparative genomics of eme..."


In [78]:
# Shows that both the 'NEW_Tip_ID' and 'Species' columns have completely the same species names
species_table[
    species_table['NEW_Tip_ID'].str.lower().str.strip().str.split('_').str.slice(0, 2).str.join(' ')
    != 
    species_table['Species'].str.lower().str.strip().str.split(' ').str.slice(0, 2).str.join(' ')
]

,assembly_fullID_updated,Y1000+_ID,NEW_Tip_ID,Order,Species,Strain_Sequenced,NRRL,CBS,Other,CTH,...,Missing,Type strain,Type Species,Y1000+ Project,Genome_Acc,Reads_Acc,Citation Number Reference,Reference,Citation number for Related Pub,Related_Prior_Pub


In [79]:
cleaned_CHCs = list(SeqIO.parse(local_path + 'genome_analyses/cleaned_CHC1_OG.fasta', 'fasta'))

CHC1_fasta_species_names = []

for seq in cleaned_CHCs:
    if '|' in seq.description:     
        species_name = seq.description.split('|')[1].split('.')[0]
    else:
        species_name = seq.description.split('.')[0]
        
    if species_name.startswith('yH'):
        species_name = species_name.split('_', 1)[1]

    tokens = species_name.split('_')
    
    if '_'.join(tokens[1:3]).lower() == 'sp_nov':
        species_name = '_'.join(tokens[:3])
    else:
        species_name = '_'.join(tokens[:2])

    CHC1_fasta_species_names.append(species_name)

In [80]:
# There seems to be 2 seq descriptions that don't end in .final or .concatenated
len(cleaned_CHCs)

1157

In [81]:
# This is why
for seq in cleaned_CHCs:
    if '.concatenated' not in seq.description and '.final' not in seq.description:
        print(seq.description)

YGL206C|saccharomyces_cerevisiae.sgd
candida_inconspicua|Pichia_inconspicua


In [82]:
# IMPORTANT: Odd situation here
CHC1_fasta_species_names[674]

'hanseniaspora_1'

In [83]:
CHC1_fasta_names_series = (
    pd.Series(CHC1_fasta_species_names).str.lower().str.strip()
)

table_species = (
    species_table['Species'].str.lower().str.strip().str.replace(' ', '_', regex=False)
)

CHC1_mismatches = CHC1_fasta_names_series[~CHC1_fasta_names_series.isin(table_species)]

In [84]:
print(CHC1_mismatches)
len(CHC1_mismatches.unique())

0             agaricus_bisporus
1          coccidioides_immitis
2       cryptococcus_neoformans
3            malassezia_globosa
4        microbotryum_violaceum
                 ...           
1150    saccharomycopsis_sp_nov
1151            kodamaea_sp_nov
1152            kodamaea_sp_nov
1153           lachancea_sp_nov
1154      schwanniomyces_sp_nov
Length: 309, dtype: object


273

In [85]:
# THIS IS TEMPORARY; REMEMBER TO ACCOUNT FOR THE SP AND SP_NOV SPECIES LATER!!!
CHC1_non_sp_mismatches = []
for name in CHC1_mismatches.unique():
    if '_sp' not in name and 'sp_nov' not in name:
        CHC1_non_sp_mismatches.append(name)

print(len(CHC1_non_sp_mismatches))
CHC1_non_sp_mismatches

244


['agaricus_bisporus',
 'coccidioides_immitis',
 'cryptococcus_neoformans',
 'malassezia_globosa',
 'microbotryum_violaceum',
 'mixia_osmundae',
 'phanerochaete_chrysosporium',
 'postia_placenta',
 'puccinia_graminis',
 'saitoella_complicata',
 'scleroderma_citrinum',
 'ustilago_maydis',
 'arthrobotrys_oligospora',
 'arxula_adeninivorans',
 'aspergillus_nidulans',
 'botrytis_cinerea',
 'candida_apicola',
 'candida_infanticola',
 'candida_versatilis',
 'fusarium_graminearum',
 'hansenula_polymorpha',
 'komagataella_phaffi',
 'lachancea_fantastica',
 'nadsonia_fulvescens',
 'neurospora_crassa',
 'saprochaete_clavata',
 'schizosaccharomyces_pombe',
 'sclerotinia_sclerotiorum',
 'stagonospora_nodorum',
 'xylona_heveae',
 'kazachstania_jainica',
 'kazachstania_servazzi',
 'candida_humilis',
 'kazachstania_transvaalensis',
 'kazachstania_yakushimaensis',
 'brettanomyces_mucatilis',
 'kazachstania_humatica',
 'kloeckera_taiwanica',
 'dipodascus_anamola',
 'myxozyma_melbiosi',
 'new_genus',
 'c

In [86]:
# Comparing assembly_fullID_updated column with the CHC1 FASTA seq descriptions to confirm that it's generally
# a good reference point for old species names
CHC1_non_sp_mismatches = pd.Series(CHC1_non_sp_mismatches)

table_assembly_fullID = (species_table['assembly_fullID_updated'].str.lower())

for species_name in CHC1_non_sp_mismatches:
    if not table_assembly_fullID.str.contains(species_name, regex=False).any():
        print(species_name)

# Only like 15 that either don't match with the values in this column, or don't exist at all. Either way, it's easy to handle these later

# UPDATE: THESE HAVE BEEN RESOLVED; THEY WERE EITHER IN THE OUTGROUP OR SIMPLY WERE UNACCOUNTED OUTDATED NAMES.

agaricus_bisporus
coccidioides_immitis
cryptococcus_neoformans
malassezia_globosa
microbotryum_violaceum
mixia_osmundae
phanerochaete_chrysosporium
postia_placenta
puccinia_graminis
saitoella_complicata
scleroderma_citrinum
ustilago_maydis
arthrobotrys_oligospora
aspergillus_nidulans
botrytis_cinerea
fusarium_graminearum
neurospora_crassa
schizosaccharomyces_pombe
sclerotinia_sclerotiorum
stagonospora_nodorum
xylona_heveae
pichia_inconspicua


#### To-Do Regarding Handling These Species:
- Remove the species that exist in the itol_color_strip file but not the species_table.csv.
- Omit any sp or sp_nov species names in the automatic name-updating, and manually update their names after.
- Handle any seqs not updated due to the fact that I modified their descriptions (e.g., .concatenated seqs).

In [87]:
# Quick confirmation
print(CHC1_non_sp_mismatches[CHC1_non_sp_mismatches.str.contains('_sp')])
print(CHC1_non_sp_mismatches[CHC1_non_sp_mismatches.str.contains('nov')])

Series([], dtype: object)
Series([], dtype: object)


In [88]:
CHC1_non_sp_mismatches[:5]

0          agaricus_bisporus
1       coccidioides_immitis
2    cryptococcus_neoformans
3         malassezia_globosa
4     microbotryum_violaceum
dtype: object

In [89]:
species_table[species_table['assembly_fullID_updated'].str.contains('arxula')]

,assembly_fullID_updated,Y1000+_ID,NEW_Tip_ID,Order,Species,Strain_Sequenced,NRRL,CBS,Other,CTH,...,Missing,Type strain,Type Species,Y1000+ Project,Genome_Acc,Reads_Acc,Citation Number Reference,Reference,Citation number for Related Pub,Related_Prior_Pub
207,arxula_adeninivorans,arxula_adeninivorans,Blastobotrys_adeninivorans_LS3,Dipodascales,Blastobotrys adeninivorans,LS3,NaN,NaN,LS3,NaN,...,3.28%,Not Type,No,No,CBZY000000000,NaN,199.0,"Kunze G, et al. The complete genome of Blastob...",,NaN


In [90]:
species_table[species_table['assembly_fullID_updated'].str.contains('arxula')].index[0]

np.int64(207)

In [91]:
species_table[species_table['assembly_fullID_updated'].str.contains(CHC1_non_sp_mismatches[0])]

,assembly_fullID_updated,Y1000+_ID,NEW_Tip_ID,Order,Species,Strain_Sequenced,NRRL,CBS,Other,CTH,...,Missing,Type strain,Type Species,Y1000+ Project,Genome_Acc,Reads_Acc,Citation Number Reference,Reference,Citation number for Related Pub,Related_Prior_Pub


In [92]:
# Making sure that there are no duplicate species names in assembly_fullID_updated
name_freq_in_IDs = {}

for species_name in CHC1_non_sp_mismatches:
    count = species_table['assembly_fullID_updated'].str.lower().str.contains(species_name, na=False).sum()
    name_freq_in_IDs[species_name] = int(count)

frequent_names = []
for name, freq in name_freq_in_IDs.items():
    if freq > 1:
        frequent_names.append(name)

frequent_names
# THESE SPECIES HAVE DUPLICATE SEQUENCES. ADDRESS THESE.

# UPDATE: THESE HAVE BEEN ADDRESSED; SOME WERE ALREADY HANDLED, AND SOME DUPES WERE NOW REMOVED ACCORDINGLY.

['candida_apicola',
 'candida_infanticola',
 'candida_versatilis',
 'komagataella_phaffi',
 'nadsonia_fulvescens',
 'candida_bombi',
 'schwanniomyces_occidentalis',
 'metschnikowia_bicuspidata',
 'dipodascopsis_uninucleata',
 'hanseniaspora_occidentalis',
 'schwanniomyces_vanrijiae',
 'metschnikowia_matae',
 'schwanniomyces_polymorphus']

In [93]:
# Omitting them for now:
for name in frequent_names:
    CHC1_non_sp_mismatches = CHC1_non_sp_mismatches[CHC1_non_sp_mismatches != name]

In [94]:
len(CHC1_non_sp_mismatches)

231

In [95]:
# Omitting species names that aren't found precisely in the IDs:
non_ID_names = []
for name, freq in name_freq_in_IDs.items():
    if freq == 0:
        non_ID_names.append(name)

non_ID_names

# THESE HAVE BEEN ADDRESSED (THEY'RE THE SAME 22 SPECIES FROM A FEW CELLS ABOVE)

['agaricus_bisporus',
 'coccidioides_immitis',
 'cryptococcus_neoformans',
 'malassezia_globosa',
 'microbotryum_violaceum',
 'mixia_osmundae',
 'phanerochaete_chrysosporium',
 'postia_placenta',
 'puccinia_graminis',
 'saitoella_complicata',
 'scleroderma_citrinum',
 'ustilago_maydis',
 'arthrobotrys_oligospora',
 'aspergillus_nidulans',
 'botrytis_cinerea',
 'fusarium_graminearum',
 'neurospora_crassa',
 'schizosaccharomyces_pombe',
 'sclerotinia_sclerotiorum',
 'stagonospora_nodorum',
 'xylona_heveae',
 'pichia_inconspicua']

In [96]:
# Omitting them for now:
for name in non_ID_names:
    CHC1_non_sp_mismatches = CHC1_non_sp_mismatches[CHC1_non_sp_mismatches != name]

In [97]:
len(CHC1_non_sp_mismatches)

209

In [98]:
old_to_new_dict = {}

for species_name in CHC1_non_sp_mismatches:
    if species_table['assembly_fullID_updated'].str.contains(species_name, na=False).any():
        row = species_table[species_table['assembly_fullID_updated'].str.contains(species_name)].index[0]
        old_to_new_dict[species_name] = species_table.loc[row, 'Species']

In [99]:
old_to_new_dict

{'arxula_adeninivorans': 'Blastobotrys adeninivorans',
 'hansenula_polymorpha': 'Ogataea polymorpha',
 'lachancea_fantastica': 'Lachancea fantastica nom. nud.',
 'saprochaete_clavata': 'Magnusiomyces clavatus',
 'kazachstania_jainica': 'Grigorovia jiainica',
 'kazachstania_servazzi': 'Kazachstania servazzii',
 'candida_humilis': 'Kazachstania humilis',
 'kazachstania_transvaalensis': 'Grigorovia transvaalensis',
 'kazachstania_yakushimaensis': 'Grigorovia yakushimaensis',
 'brettanomyces_mucatilis': 'Botryozyma mucatilis',
 'kazachstania_humatica': 'Grigorovia humatica',
 'kloeckera_taiwanica': 'Hanseniaspora taiwanica',
 'dipodascus_anamola': 'Dipodascopsis anomala',
 'myxozyma_melbiosi': 'Myxozyma melibiosi',
 'candida_pseudorugosa': 'Diutina pseudorugosa',
 'geotrichum_klebahnii': 'Dipodascus klebahnii',
 'candida_cellae': 'Starmerella cellae',
 'candida_etchellsii': 'Starmerella etchellsii',
 'candida_floricola': 'Starmerella floricola',
 'candida_floris': 'Starmerella floris',
 'c

In [100]:
# Proves there are no duplicate names in the CHC1 orthogroup file for the species in the dictionary:
cleaned_CHCs = list(SeqIO.parse(local_path + 'genome_analyses/cleaned_CHC1_OG.fasta', 'fasta'))

for old_name, new_name in old_to_new_dict.items():
    count = 0
    for record in cleaned_CHCs:
        if old_name in record.description:
            count += 1

    if count > 1:
        print(old_name, count)

# Temporarily handled the potential paralogs by somewhat randomly deleting 1 of the 2 seqs

In [101]:
import copy

cleaned_CHCs_copy = copy.deepcopy(cleaned_CHCs)

for old_name, new_name in old_to_new_dict.items():
    for record in cleaned_CHCs_copy:
        if old_name in record.description.lower(): # I THINK I confirmed that there are no duplicates in the FASTA, but confirm.
            if 'updated' not in record.description.lower():
                appended_name = new_name.replace(' ', '_').lower().strip()
                #record.description = record.description + f'|updated={appended_name}'
                record.description = f'|updated={appended_name}'
            break

out_path = local_path + 'genome_analyses/upd_cleaned_CHC1_OG.fasta'

with open(out_path, 'w') as f:
    SeqIO.write(cleaned_CHCs_copy, f, 'fasta')

print('Saved to:', out_path)

Saved to: /Users/georgecrawford/Documents/yeast-clathrin-conservation/genome_analyses/upd_cleaned_CHC1_OG.fasta


#### Notes:
- **UNRESOLVED** Handle the recently-identified species with duplicate seqs.
- **UNRESOLVED** Handle the recently-identified species that don't appear in the assembly ID column.

In [102]:
# Get rid of strain specifications (confirm fully)
species_clade_df['species'] = species_clade_df['species'].apply(lambda x: '_'.join(x.split('_')[:2]))
species_clade_df.iloc[512]

species        Saturnispora_diversa
clade_color                 #FF8200
clade_name                Pichiales
Name: 512, dtype: object

In [103]:
# Get rid of duplicate species rows now that strain specifications are gone (confirm fully)
species_clade_df = species_clade_df.drop_duplicates(subset=['species'])

In [104]:
species_clade_df

,species,clade_color,clade_name
0,Nakazawaea_anatomiae,#6A3906,Alaninetales
1,Nakazawaea_ernobii,#6A3906,Alaninetales
2,Nakazawaea_holstii,#6A3906,Alaninetales
3,Nakazawaea_ishiwadae,#6A3906,Alaninetales
4,Nakazawaea_laoshanensis,#6A3906,Alaninetales
...,...,...,...
1091,Tortispora_starmeri,#FF0090,Trigonopsidales
1092,Trigonopsis_californica,#FF0090,Trigonopsidales
1093,Trigonopsis_cantarellii,#FF0090,Trigonopsidales
1094,Trigonopsis_variabilis,#FF0090,Trigonopsidales


In [106]:
species_clade_df[species_clade_df['clade_name'].isna()]

,species,clade_color,clade_name
248,Agaricus_bisporus,#050f07,NaN
249,Aspergillus_nidulans,#050f07,NaN
250,Botrytis_cinerea,#050f07,NaN
251,Coccidioides_immitis,#050f07,NaN
252,Cryptococcus_neoformans,#050f07,NaN
253,Fusarium_graminearum,#050f07,NaN
254,Malassezia_globosa,#050f07,NaN
255,Microbotryum_violaceum,#050f07,NaN
256,Mixia_osmundae,#050f07,NaN
257,Neurospora_crassa,#050f07,NaN


In [107]:
# Give the outgroup species an "Outgroup" clade name
species_clade_df['clade_name'] = species_clade_df['clade_name'].fillna('Outgroup')

In [108]:
species_clade_df[species_clade_df['clade_name'].isna()]

,species,clade_color,clade_name


In [169]:
upd_cleaned_CHCs = list(SeqIO.parse(local_path + 'genome_analyses/upd_cleaned_CHC1_OG.fasta', 'fasta'))

for record in upd_cleaned_CHCs:
    record.description = record.description.replace(' ', '')

In [170]:
print(upd_cleaned_CHCs[14].description)

g002281.m1|arxula_adeninivorans.final|updated=blastobotrys_adeninivorans


In [171]:
# Append clade names to each seq entry
df_species_list = species_clade_df['species'].str.lower().tolist()

for record in upd_cleaned_CHCs:
    if '|clade=' in record.description:
        continue
    
    if 'updated=' in record.description:
        updated_name = record.description.split('|updated=')[1].strip() # Confirm
        
        if any(name == updated_name.lower() for name in df_species_list):
            row = species_clade_df[species_clade_df['species'].str.lower() == updated_name.lower()]
            clade_name = row['clade_name'].values[0]
            record.description = record.description + f'|clade={clade_name}'
            
    elif any(name in record.description.lower() for name in df_species_list):
        matched_name = next(name for name in df_species_list if name in record.description.lower())
        mask = species_clade_df['species'].str.lower() == matched_name
        clade_name = species_clade_df.loc[mask, 'clade_name'].iloc[0]
        record.description = record.description + f'|clade={clade_name}'

    record.id = record.description

In [172]:
print(upd_cleaned_CHCs[14].description)

g002281.m1|arxula_adeninivorans.final|updated=blastobotrys_adeninivorans|clade=Dipodascales


In [173]:
upd_cleaned_CHCs[255].description

'g003850.m1|yHMPu5000034624_pichia_nakasei_180604.final|clade=Pichiales'

In [174]:
# All the species and/or FASTA entries that currently do NOT have an assigned clade
# Much of the species here were part of groups I purposely removed before running the code directly above 
# Thus, refer to my earlier code where I removed these groups when addressing these entires
for record in upd_cleaned_CHCs:
    if 'clade=' not in record.description:
        print(record.description)

g001204.m1|Postia_placenta.final
g005968.m1|arthrobotrys_oligospora.final
g000288.m1|candida_apicola.final
g000343.m1|candida_infanticola.final
g000662.m1|candida_versatilis.final
g001350.m1|komagataella_phaffi.final
g001279.m1|lachancea_fantastica.final|updated=lachancea_fantastica_nom._nud.
g010820.m1|stagonospora_nodorum.final
g005428.m1|yHDO568_nakaseomyces_sp_190924.final
g003331.m1|yHDO569_candida_sp_180604.final
g002270.m1|yHDO576_kazachstania_sp_180604.final
g000614.m1|yHDO578_saturnispora_sp_180604.final
g004913.m1|yHDO592_zygotorulaspora_sp_190924.final
g005281.m1|yHDO593_barnettozyma_sp_180604.final
g001959.m1|yHDR128_Nakazawaea_sp_nov_plate34_PLATANUS.final
g003584.m1|yHKB15_Yamadazyma_sp_nov_plate33_SPADES.final
g001940.m1|yHKB289_Vanderwaltozyma_sp_nov_plate33_SPADES.final
g002067.m1|yHKB289_Vanderwaltozyma_sp_nov_plate33_SPADES.final
g000377.m1|yHKB357_New_Genus_SPADES.final
g001110.m1|yHKS168_Pichia_sp_nov_plate33_PLATANUS.final
g001351.m1|yHKS545_Nakazawaea_sp_nov_plat

In [175]:
upd_cleaned_CHCs[0].description

'g008776.m1|Agaricus_bisporus.final|clade=Outgroup'

In [176]:
for record in upd_cleaned_CHCs:
    if 'g0' not in record.description:
        print(record.description)

YGL206C|saccharomyces_cerevisiae.sgd|clade=Saccharomycetales
spathaspora_gorwiae.concatenated|clade=Serinetales
kazachstania_slooffiae.concatenated|clade=Saccharomycetales
kazachstania_spencerorum.concatenated|clade=Saccharomycetales
candida_digboiensis.concatenated|clade=Dipodascales
myxozyma_neglecta.concatenated|clade=Lipomycetales
Candida_dendrica.concatenated|clade=Phaffomycetales
Magnusiomyces_magnusii.concatenated|clade=Dipodascales
candida_metapsilosis.concatenated|clade=Serinetales
wickerhamomyces_chambardii.concatenated|clade=Phaffomycetales
candida_saitoana.concatenated|clade=Serinetales
cyberlindnera_mississippiensis.concatenated|clade=Phaffomycetales
zygosaccharomyces_pseudobailii.concatenated|clade=Saccharomycetales
starmera_amethionina.concatenated|clade=Phaffomycetales
nakaseomyces_bacillisporus.concatenated|clade=Saccharomycetales
Wickerhamomyces_kurtzmanii.concatenated|clade=Phaffomycetales
metschnikowia_fructicola.concatenated|clade=Serinetales
ascoidea_tarda.concate

In [177]:
upd_cleaned_CHCs_dupes = []

for i in range(len(upd_cleaned_CHCs)):
    species_name_parts = upd_cleaned_CHCs[i].description.split('|')
    for part in species_name_parts:
        if 'updated=' in part:
            species_name = part.replace('updated=', '')
            break
        elif '_' in part:
            species_name = part.split('.')[0]
    
    # species_name = upd_cleaned_CHCs[i].description.split('|')[1].split('.')[0]
    
    if species_name.startswith('yH'):
        species_name = species_name.split('_', 1)[1]
        
    for j in range(len(upd_cleaned_CHCs)):
        if i == j:
            continue
            
        #other_species_name = upd_cleaned_CHCs[j].description.split('|')[1].split('.')[0]

        other_species_name_parts = upd_cleaned_CHCs[j].description.split('|')
        for part in other_species_name_parts:
            if 'updated=' in part:
                other_species_name = part.replace('updated=', '')
            elif '_' in part:
                other_species_name = part.split('.')[0]
        
        if other_species_name.startswith('yH'):
            other_species_name = other_species_name.split('_', 1)[1]
        
        if species_name.lower() == other_species_name.lower() and species_name.lower() not in upd_cleaned_CHCs_dupes:
            upd_cleaned_CHCs_dupes.append(species_name.lower())
            print(f'Duplicate sequences found for {species_name}')

Duplicate sequences found for brettanomyces_anomalus
Duplicate sequences found for kazachstania_humilis
Duplicate sequences found for Vanderwaltozyma_sp_nov_plate33_SPADES
Duplicate sequences found for Candida_sp
Duplicate sequences found for pichia_sp_170307
Duplicate sequences found for tetrapisispora_blattae_190924
Duplicate sequences found for candida_sp_160519
Duplicate sequences found for clavispora_fructus
Duplicate sequences found for yueomyces_sinensis_180604
Duplicate sequences found for candida_sp_170912
Duplicate sequences found for dipodascus_aggregatus_190924
Duplicate sequences found for candida_glucosophila_180604
Duplicate sequences found for pichia_sp_180604


In [178]:
with open(local_path + 'genome_analyses/upd_cleaned_w_clades_CHCs.fasta', 'w') as handle:
    SeqIO.write(upd_cleaned_CHCs, handle, 'fasta')

In [193]:
# Dictionary of all the outdated names corresponding to updated names that I found manually.

manually_upd_names = {
    'candida_apicola': 'starmerella_apicola',
    'candida_infanticola': 'wickerhamiella_infanticola',
    'candida_versatilis': 'wickerhamiella_versatilis',
    'yHDO568_nakaseomyces_sp_190924': 'Nakaseomyces_sp._yHDO568',
    'yHDO569_candida_sp_180604': 'Candida_sp._yHDO569',
    'yHDO576_kazachstania_sp_180604': 'Kazachstania_sp._UFMG-CM-Y273',
    'yHDO578_saturnispora_sp_180604': 'Saturnispora_bothae',
    'yHDO592_zygotorulaspora_sp_190924': 'Zygotorulaspora_sp._yHDO592',
    'yHDO593_barnettozyma_sp_180604': 'Barnettozyma_sp._yHDO593',
    'yHDR128_Nakazawaea_sp_nov_plate34_PLATANUS': 'Nakazawaea_sp._yHDR128',
    'yHKB15_Yamadazyma_sp_nov_plate33_SPADES': 'Yamadazyma_sp._yHKB15',
    'yHKB289_Vanderwaltozyma_sp_nov_plate33_SPADES': 'Vanderwaltozyma_sp._yHKB289',
    'yHKB357_New_Genus_SPADES': 'Candida_sp._yHKB357',
    'yHKS168_Pichia_sp_nov_plate33_PLATANUS': 'Pichia_sp._yHKS168',
    'yHKS545_Nakazawaea_sp_nov_plate33_PLATANUS': 'Nakazawaea_sp._yHKS545',
    'yHKS617_Sugiyamaella_sp_nov_plate33_SPADES': 'Sugiyamaella_sp._yHKS617',
    'yHKS641_Suhomyces_sp_nov_plate33_SPADES': 'Suhomyces_sp._yHKS641',
    'yHMH26_Wickerhamomyces_sp_nov_plate34_DISCOVAR': 'Wickerhamomyces_sp._yHMH26',
    'yHMH407_Schwanniomyces_sp_nov_plate33_DISCOVAR': 'Schwanniomyces_sp._yHMH407',
    'yHMH443_Groenewaldozyma_sp_nov_plate33_SPADES': 'Groenewaldozyma_sp._yHMH443',
    'yHMH446_Pichia_sp_nov_plate33_SPADES': 'Pichia_galeolata',
    'yHMH451_Wickerhamomyces_sp_nov_plate33_PLATANUS': 'Wickerhamomyces_sp._yHMH451',
    'yHMH617_Wickerhamomyces_sp_nov_plate34_PLATANUS': 'Wickerhamomyces_sp._yHMH617',
    'yHMH660_Kluyveromyces_sp_nov_plate33_SPADES': 'Kluyveromyces_sp._yHMH660',
    'yHMJ1_Ogataea_sp_nov_plate33_PLATANUS': 'Ogataea_sp._yHMJ1',
    'yHMJ407_Torulaspora_sp_nov_plate34_SPADES': 'Torulaspora_sp._yHMJ407',
    'yHMJ9_Metschnikowia_sp_nov_plate34_DISCOVAR': 'Metschnikowia_sp._yHJM9',
    'yHMPu5000026157_candida_sp_170713': 'Wickerhamomyces_sp._NRRL_Y-7574',
    'yHMPu5000026190_ogataea_sp_160519': 'Ogataea_sp._NRRL_YB-2437',
    'yHMPu5000034596_candida_sp_160613': 'Starmera_sp._NRRL_Y-17713',
    'yHMPu5000034620_pichia_sp_170307': 'Pichia_sp._NRRL_Y-12830',
    'yHMPu5000034621_pichia_sp_160519': 'Pichia_sp._NRRL_Y-12827',
    'yHMPu5000034669_Blastobotrys_raffinofermentans_SPAdes': 'Blastobotrys_raffinosifermentans',
    'yHMPu5000034744_lipomyces_spencer-martinsiae_170307': 'Lipomyces_sp._NRRL_Y-7042',
    'yHMPu5000034745_lipomyces_sp_201018': 'Lipomyces_sp._NRRL_Y-27488',
    'yHMPu5000034908_candida_sp_160519': 'Ogataea_sp._NRRL_YB-2442',
    'yHMPu5000034909_candida_sp_160519': 'Ogataea_sp._NRRL_Y-27170',
    'yHMPu5000034910_candida_sp_160519': 'Ogataea_sp._NRRL_YB-1238',
    'yHMPu5000034969_candida_sp_160519.haplomerger2': 'Cyberlindnera_sp._NRRL_Y-27103',
    'yHMPu5000034971_candida_sp_160519': 'Cyberlindnera_sp._NRRL_Y-7615',
    'yHMPu5000035020_Candida_chickasaworum_SPADES': 'Suhomyces_chickasaworum',
    'yHMPu5000035024_Candida_kunorum_SPADES': 'Suhomyces_kunorum',
    'yHMPu5000035026_Candida_anneliseae_SPADES': 'Suhomyces_anneliseae',
    'yHMPu5000035030_Candida_lycoperdinae__SPADES': 'Teunomyces_lycoperdinae',
    'yHMPu5000035031_Candida_kruisii_SPADES': 'Teunomyces_kruisii',
    'yHMPu5000035265_Candida_sp_SPADES': 'Candida_sp._NRRL_Y-27127',
    'yHMPu5000035285_Candida_alocasiicola_SPADES': 'Wickerhamiella_alocasiicola',
    'yHMPu5000035299_pichia_sp_170307': 'Pichia_sp._NRRL_Y-12824',
    'yHMPu5000035300_pichia_sp_170307': 'Pichia_sp._NRRL_YB-4149',
    'yHMPu5000035319_Candida_tartarivorans_S177_SPADES': 'Groenewaldozyma_tartarivorans',
    'yHMPu5000037240_candida_sp_210210': 'Candida_margitis',
    'yHMPu5000037243_lodderomyces_sp_210210': 'Lodderomyces_beijingensis',
    'yHMPu5000037857_scheffersomyces_spartiniae_170210': 'Scheffersomyces_spartinae',
    'yHMPu5000037870_candida_sp_170912': 'Cyberlindnera_sp._NRRL_Y-27267',
    'yHMPu5000037871_Candida_trypodendroni_SPADES': 'Candida_trypodendri',
    'yHMPu5000037879_Lipomyces_spencer-martinsiae_S172_SPADES': 'Lipomyces_spencermartinsiae',
    'yHMPu5000038053_pichia_sp_180604': 'Pichia_sp._NRRL_Y-27261',
    'yHMPu5000038076_Candida_tumulicola_SPADES': 'Yamadazyma_tumulicola',
    'yHMPu5000038085_candida_sp_170912': 'Wickerhamomyces_sp._NRRL_YB-3031',
    'yHMPu5000038097_candida_sp_201018': 'Ogataea_sp._NRRL_Y-27166',
    'yHMPu5000038336_pichia_sp_170713': 'Pichia_sp._NRRL_Y-27259',
    'Candida_jeffriesii': 'Spathaspora_jeffriesii',
    'yHMPu5000041785_saccharomycopsis_sp_180604': 'Saccharomycopsis_sp._NRRL_Y-5750',
    'Metschnikowia_zizyphicola': 'Metschnikowia_ziziphicola',
    'yHQL14_Wickerhamomyces_sp_nov_plate33_MASURCA': 'Wickerhamomyces_sp._yHQL14',
    'yHQL182_Saccharomycopsis_sp_nov_plate33_SPADES': 'Saccharomycopsis_sp._yHQL182',
    'yHQL449_Kodamaea_sp_nov_plate33_PLATANUS': 'Kodamaea_sp._yHQL449',
    'yHQL451_Kodamaea_sp_nov_plate33_SPADES': 'Kodamaea_sp._yHQL451',
    'yHQL494_Lachancea_sp_nov_plate34_PLATANUS': 'Lachancea_sp._yHQL494',
    'yHRVM31_Schwanniomyces_sp_nov_plate33_SPADES': 'Schwanniomyces_sp._NRRL_Y-7430'
}

In [194]:
final_CHCs = list(SeqIO.parse(local_path + 'genome_analyses/FINAL_upd_cleaned_w_clades_CHCs.fasta', 'fasta'))

In [195]:
import copy

final_CHCs_copy = copy.deepcopy(final_CHCs)

for old_name, new_name in manually_upd_names.items():
    for record in final_CHCs_copy:
        if old_name.lower() in record.description.lower():
            if 'updated' not in record.description.lower() and 'clade' not in record.description.lower():
                appended_name = new_name.lower().strip()
                
                record.description = record.description + f'|updated={appended_name}'
                record.id = record.description
                
            break

In [203]:
non_clade_counter = 0
for record in final_CHCs_copy:
    if 'clade' not in record.description:
        non_clade_counter += 1

print(non_clade_counter)

72


In [204]:
# Re-establish the species_clade_df WITH strain specifications for this new batch of clade assignments.

colorstrip_path = local_path + 'fig2_tree_data/itol_color_strip.txt'

with open(colorstrip_path, 'r') as f:
    lines = f.readlines()

data_rows = []
for line in lines[42:]:
    line = line.strip()
    if not line:
        continue
    parts = line.split()
    if len(parts) != 3:
        raise ValueError(f'Unexpected number of columns in line: {line}')
    data_rows.append(parts)


species_strain_clade_df = pd.DataFrame(data_rows, columns=['species', 'fill_color', 'border_color'])

species_strain_clade_df = species_strain_clade_df.apply(lambda x: x.str.strip())

print(species_strain_clade_df.shape)
species_strain_clade_df

(1175, 3)


,species,fill_color,border_color
0,Nakazawaea_anatomiae,#6A3906,#6A3906
1,Nakazawaea_ernobii,#6A3906,#6A3906
2,Nakazawaea_holstii,#6A3906,#6A3906
3,Nakazawaea_ishiwadae,#6A3906,#6A3906
4,Nakazawaea_laoshanensis,#6A3906,#6A3906
...,...,...,...
1170,Tortispora_starmeri,#FF0090,#FF0090
1171,Trigonopsis_californica,#FF0090,#FF0090
1172,Trigonopsis_cantarellii,#FF0090,#FF0090
1173,Trigonopsis_variabilis,#FF0090,#FF0090


In [205]:
species_strain_clade_df = species_strain_clade_df.drop(columns='border_color').rename(columns={'fill_color': 'clade_color'})
species_strain_clade_df

,species,clade_color
0,Nakazawaea_anatomiae,#6A3906
1,Nakazawaea_ernobii,#6A3906
2,Nakazawaea_holstii,#6A3906
3,Nakazawaea_ishiwadae,#6A3906
4,Nakazawaea_laoshanensis,#6A3906
...,...,...
1170,Tortispora_starmeri,#FF0090
1171,Trigonopsis_californica,#FF0090
1172,Trigonopsis_cantarellii,#FF0090
1173,Trigonopsis_variabilis,#FF0090


In [206]:
for line in lines:
    if line.startswith('LEGEND_COLORS'):
        colors = line.strip().split()[1:]
    if line.startswith('LEGEND_LABELS'):
        labels = line.strip().split()[1:]

color_to_clade = dict(zip(colors, labels))

species_strain_clade_df['clade_name'] = species_strain_clade_df['clade_color'].map(color_to_clade)

In [213]:
# It now includes strain specifications
species_strain_clade_df.loc[11]

species        Nakazawaea_sp._yHKS545
clade_color                   #6A3906
clade_name               Alaninetales
Name: 11, dtype: object

In [215]:
# Append clade names to each seq entry
df_species_list = species_strain_clade_df['species'].str.lower().tolist()

for record in final_CHCs_copy:
    if 'clade=' in record.description:
        continue
    
    if 'updated=' in record.description:
        updated_name = record.description.split('|updated=')[1].strip() # Confirm
        
        if any(name == updated_name.lower() for name in df_species_list):
            row = species_strain_clade_df[species_strain_clade_df['species'].str.lower() == updated_name.lower()]
            clade_name = row['clade_name'].values[0]
            record.description = record.description + f'|clade={clade_name}'

    else:
        cleaned_desc = record.description.lower().replace('.final', '').replace('.concatenated', '').strip()
       
        desc_parts = cleaned_desc.split('|')

        if len(desc_parts) > 1:
            species_name = desc_parts[1].strip()
        else:
            species_name = desc_parts[0].strip()

        if any(name == species_name for name in df_species_list):
            row = species_strain_clade_df[species_strain_clade_df['species'].str.lower() == species_name]
            clade_name = row['clade_name'].values[0]
            record.description = record.description + f'|clade={clade_name}'
            
    #elif any(name in record.description.lower() for name in df_species_list):
    #    matched_name = next(name for name in df_species_list if name in record.description.lower())
    #    mask = species_strain_clade_df['species'].str.lower() == matched_name
    #    clade_name = species_strain_clade_df.loc[mask, 'clade_name'].iloc[0]
    #    record.description = record.description + f'|clade={clade_name}'

    record.id = record.description

In [220]:
non_clade_counter = 0
for record in final_CHCs_copy:
    if 'clade' not in record.description:
        non_clade_counter += 1
        print(record.description)

print(f'Number of species still with no assigned clade: {non_clade_counter}')

g000288.m1|candida_apicola.final|updated=starmerella_apicola
g000343.m1|candida_infanticola.final|updated=wickerhamiella_infanticola
g000662.m1|candida_versatilis.final|updated=wickerhamiella_versatilis
Number of species still with no assigned clade: 3


In [222]:
out_path = local_path + 'genome_analyses/FULL_FINAL_CHCs.fasta'

with open(out_path, 'w') as f:
    SeqIO.write(final_CHCs_copy, f, 'fasta')

print('Saved to:', out_path)

Saved to: /Users/georgecrawford/Documents/yeast-clathrin-conservation/genome_analyses/FULL_FINAL_CHCs.fasta


In [189]:
for record in upd_cleaned_CHCs:
    if len(record.seq) <= 1620:
        print(record.description, len(record.seq))

spathaspora_gorwiae.concatenated|clade=Serinetales 1518
g010820.m1|stagonospora_nodorum.final 1604
candida_digboiensis.concatenated|clade=Dipodascales 1400
g003699.m1|yHMPu5000037231_lipomyces_yamanashiensis_210210.final|clade=Lipomycetales 1575
g000799.m1|yHMPu5000037870_candida_sp_170912.final 1402
g005122.m1|yHMPu5000037870_candida_sp_170912.final 326
g000137.m1|yHMPu5000038053_pichia_sp_180604.final 1216
g000584.m1|yHMPu5000038053_pichia_sp_180604.final 377
g001718.m1|yHMPu5000038053_pichia_sp_180604.final 1201
g001720.m1|yHMPu5000038053_pichia_sp_180604.final 387
candida_insectamans.concatenated|updated=hemisphaericaspora_insectamans|clade=Serinetales 1563
Candida_jeffriesii.concatenated 1578
Metschnikowia_zizyphicola.concatenated 1505
metschnikowia_chrysoperlae.concatenated|clade=Serinetales 1613
g004647.m1|yHMPu5000041819_magnusiomyces_starmeri_201018.final|clade=Dipodascales 1582
dipodascus_armillariae.concatenated|clade=Dipodascales 1504


In [ ]:
# IMPORTANT: HANDLE THESE OUTLIERS BEFORE GENERATING THE GRAPHS!!!

#### Finishing CLC Cleaning

In [228]:
cleaned_CLCs = list(SeqIO.parse(local_path + 'genome_analyses/cleaned_CLC1_OG.fasta', 'fasta'))

print('Outliers to be handled:')
for record in cleaned_CLCs:
    if len(record.seq) < 150:
        print(len(record.seq))
    elif len(record.seq) > 300:
        print(len(record.seq))

Outliers to be handled:
319
301
149
146
134
760
125
125
125
1503
1637
60
49
125
125
125
125
125
125
125
125
125


In [237]:
# All of the species names in New_Tip_ID are the exact same as those in the clade_df
same_values = species_table['NEW_Tip_ID'].isin(species_strain_clade_df['species']).all()
print(same_values)

True


In [238]:
fasta_names_list = []

for record in cleaned_CLCs:
    cleaned_desc = record.description.lower().replace('.final', '').replace('.concatenated', '').strip()
    desc_parts = cleaned_desc.split('|')

    if len(desc_parts) > 1:
        species_name = desc_parts[1].strip()
    else:
        species_name = desc_parts[0].strip()

    fasta_names_list.append(species_name)

fasta_names_series = pd.Series(fasta_names_list)

same_CLC_names = fasta_names_series.isin(species_table['assembly_fullID_updated'].str.lower()).all()
print(same_CLC_names)

False


In [249]:
assembly_fullID_updated_list = species_table['assembly_fullID_updated'].tolist()

for name in fasta_names_list:
    found_match = False
    for entry in assembly_fullID_updated_list:
        if entry.strip().lower() == name.strip().lower():
            found_match = True

    if not found_match:
        print(f'No match found for {name}')

No match found for agaricus_bisporus
No match found for coccidioides_immitis
No match found for cryptococcus_neoformans
No match found for malassezia_globosa
No match found for microbotryum_violaceum
No match found for mixia_osmundae
No match found for phanerochaete_chrysosporium
No match found for postia_placenta
No match found for puccinia_graminis
No match found for saitoella_complicata
No match found for scleroderma_citrinum
No match found for ustilago_maydis
No match found for arthrobotrys_oligospora
No match found for aspergillus_nidulans
No match found for botrytis_cinerea
No match found for candida_albicans.cgd
No match found for fusarium_graminearum
No match found for neurospora_crassa
No match found for saccharomyces_cerevisiae.sgd
No match found for schizosaccharomyces_pombe
No match found for sclerotinia_sclerotiorum
No match found for stagonospora_nodorum
No match found for xylona_heveae
No match found for magnusiomyces_magnusii
No match found for pichia_inconspicua


In [253]:
# STEPS TO FOLLOW BELOW:

# 1. Get rid of duplicate species entries
# 2. Update ALL species names (it's easier doing this now with CLC than with CHC)
# 3. Update names directly above this that didn't get a match (if necessary)
# 4. Assign clades

In [261]:
# Update names for all species (even if some don't really need to be updated, we know the NEW_Tip_ID values
# are the same as the ones in the species_strain_clade_df 'species' column).

for record in cleaned_CLCs:
    if 'updated=' in record.description:
        continue
        
    cleaned_desc = record.description.lower().replace('.final', '').replace('.concatenated', '').strip()
    desc_parts = cleaned_desc.split('|')

    if len(desc_parts) > 1:
        species_name = desc_parts[1].strip().lower()
    else:
        species_name = desc_parts[0].strip().lower()

    mask = species_table['assembly_fullID_updated'].str.strip().str.lower() == species_name

    match_count = mask.sum()

    if match_count == 1:
        updated_name = species_table.loc[mask, 'NEW_Tip_ID'].values[0]
        record.description = record.description + f'|updated={updated_name}'
        record.id = record.description

    elif match_count > 1:
        all_matches = species_table.loc[mask, 'NEW_Tip_ID'].tolist()
        print(f"Warning: Multiple matches found for {species_name}. Matches: {all_matches}")
        
    else:
        print(f'No match found in species_table for {species_name}')

No match found in species_table for agaricus_bisporus
No match found in species_table for coccidioides_immitis
No match found in species_table for cryptococcus_neoformans
No match found in species_table for malassezia_globosa
No match found in species_table for microbotryum_violaceum
No match found in species_table for mixia_osmundae
No match found in species_table for phanerochaete_chrysosporium
No match found in species_table for postia_placenta
No match found in species_table for puccinia_graminis
No match found in species_table for saitoella_complicata
No match found in species_table for scleroderma_citrinum
No match found in species_table for ustilago_maydis
No match found in species_table for arthrobotrys_oligospora
No match found in species_table for aspergillus_nidulans
No match found in species_table for botrytis_cinerea
No match found in species_table for candida_albicans.cgd
No match found in species_table for fusarium_graminearum
No match found in species_table for neurospo

In [263]:
# Append clade names to each seq entry
df_species_list = species_strain_clade_df['species'].str.lower().tolist()

for record in cleaned_CLCs:
    if 'clade=' in record.description:
        continue
    
    if 'updated=' in record.description:
        updated_name = record.description.split('|updated=')[1].strip()
        
        if any(name == updated_name.lower() for name in df_species_list):
            row = species_strain_clade_df[species_strain_clade_df['species'].str.lower() == updated_name.lower()]
            clade_name = row['clade_name'].values[0]
            record.description = record.description + f'|clade={clade_name}'

    else:
        cleaned_desc = record.description.lower().replace('.final', '').replace('.concatenated', '').strip()
       
        desc_parts = cleaned_desc.split('|')

        if len(desc_parts) > 1:
            species_name = desc_parts[1].strip()
        else:
            species_name = desc_parts[0].strip()

        if any(name == species_name for name in df_species_list):
            row = species_strain_clade_df[species_strain_clade_df['species'].str.lower() == species_name]
            clade_name = row['clade_name'].values[0]
            record.description = record.description + f'|clade={clade_name}'

    record.id = record.description

In [264]:
non_clade_counter = 0
for record in cleaned_CLCs:
    if 'clade' not in record.description:
        non_clade_counter += 1
        print(record.description)

print(f'Number of species still with no assigned clade: {non_clade_counter}')

g012115.m1|Postia_placenta.final
g004831.m1|arthrobotrys_oligospora.final
C4_01980C_A|candida_albicans.cgd
YGR167W|saccharomyces_cerevisiae.sgd
g009764.m1|stagonospora_nodorum.final
>candida_inconspicua|Pichia_inconspicua
Number of species still with no assigned clade: 6


In [266]:
cleaned_CLCs[500].description

'g000947.m1|yHMPu5000035008_candida_vadensis_160519.final|updated=Suhomyces_vadensis|clade=Serinetales'

In [267]:
out_path = local_path + 'genome_analyses/FULL_FINAL_CLCs.fasta'

with open(out_path, 'w') as f:
    SeqIO.write(cleaned_CLCs, f, 'fasta')

print('Saved to:', out_path)

Saved to: /Users/georgecrawford/Documents/yeast-clathrin-conservation/genome_analyses/FULL_FINAL_CLCs.fasta


In [297]:
full_final_CLCs = list(SeqIO.parse(local_path + 'genome_analyses/FULL_FINAL_CLCs.fasta', 'fasta'))

CLC_names = []

for record in full_final_CLCs:
    if 'updated=' in record.description:
        species_name = record.description.lower().split('|updated=')[1].split('|clade=')[0].strip()

    else:
        cleaned_desc = record.description.lower().replace('.final', '').replace('.concatenated', '').strip()
       
        desc_parts = cleaned_desc.split('|')

        if len(desc_parts) > 1:
            species_name = desc_parts[1].strip()
        else:
            species_name = desc_parts[0].strip()

    CLC_names.append(species_name)

In [298]:
from collections import Counter

counts = Counter(CLC_names)
duplicates = [string for string, count in counts.items() if count > 1]

print(duplicates)

['lipomyces_yarrowii']


In [ ]:
# Look into Lipomyces_yarrowii -- this was not resolved

In [300]:
CLC_names_no_strain = []

for record in full_final_CLCs:
    if 'updated=' in record.description:
        species_name = record.description.lower().split('|updated=')[1].split('|clade=')[0].strip()

    else:
        cleaned_desc = record.description.lower().replace('.final', '').replace('.concatenated', '').strip()
       
        desc_parts = cleaned_desc.split('|')

        if len(desc_parts) > 1:
            species_name = desc_parts[1].strip()
        else:
            species_name = desc_parts[0].strip()

    species_name_parts = species_name.split('_')
    species_name_no_strain = '_'.join(species_name_parts[:2])

    CLC_names_no_strain.append(species_name_no_strain)

In [308]:
counts = Counter(CLC_names_no_strain)
CLC_no_strain_duplicates = [string for string, count in counts.items() if count > 1]

print(CLC_no_strain_duplicates)
print(f'Number of non-strain-specific duplicate species: {len(CLC_no_strain_duplicates)}')

['alloascoidea_hylecoeti', 'blastobotrys_adeninivorans', 'brettanomyces_anomalus', 'starmerella_apicola', 'candida_auris', 'candida_dubliniensis', 'wickerhamiella_infanticola', 'candida_orthopsilosis', 'candida_tropicalis', 'wickerhamiella_versatilis', 'clavispora_lusitaniae', 'geotrichum_candidum', 'hanseniaspora_uvarum', 'ogataea_polymorpha', 'kluyveromyces_aestuarii', 'kluyveromyces_marxianus', 'komagataella_pastoris', 'komagataella_phaffii', 'metschnikowia_hawaiiensis', 'metschnikowia_similis', 'meyerozyma_guilliermondii', 'nadsonia_fulvescens', 'ogataea_parapolymorpha', 'pichia_kudriavzevii', 'scheffersomyces_stipitis', 'kazachstania_humilis', 'candida_sp.', 'nakazawaea_sp.', 'yamadazyma_sp.', 'metschnikowia_sp.', 'pichia_sp.', 'suhomyces_sp.', 'wickerhamomyces_sp.', 'schwanniomyces_sp.', 'ogataea_sp.', 'lipomyces_yarrowii', 'lipomyces_sp.', 'cyberlindnera_sp.', 'clavispora_fructus', 'yarrowia_lipolytica', 'saccharomycopsis_sp.', 'kodamaea_sp.']
Number of non-strain-specific dupli

In [309]:
full_final_CHCs = list(SeqIO.parse(local_path + 'genome_analyses/FULL_FINAL_CHCs.fasta', 'fasta'))

CHC_names_no_strain = []

for record in full_final_CHCs:
    if 'updated=' in record.description:
        species_name = record.description.lower().split('|updated=')[1].split('|clade=')[0].strip()

    else:
        cleaned_desc = record.description.lower().replace('.final', '').replace('.concatenated', '').strip()
       
        desc_parts = cleaned_desc.split('|')

        if len(desc_parts) > 1:
            species_name = desc_parts[1].strip()
        else:
            species_name = desc_parts[0].strip()

    species_name_parts = species_name.split('_')
    species_name_no_strain = '_'.join(species_name_parts[:2])

    CHC_names_no_strain.append(species_name_no_strain)

counts = Counter(CHC_names_no_strain)
CHC_no_strain_duplicates = [string for string, count in counts.items() if count > 1]

print(CHC_no_strain_duplicates)
print(f'Number of non-strain-specific duplicate species: {len(CHC_no_strain_duplicates)}')

['clade=serinetales', 'clade=saccharomycetales', 'candida_sp.', 'nakazawaea_sp.', 'pichia_sp.', 'wickerhamomyces_sp.', 'schwanniomyces_sp.', 'ogataea_sp.', 'clade=dipodascales', 'clade=phaffomycetales', 'lipomyces_sp.', 'cyberlindnera_sp.', 'saccharomycopsis_sp.', 'kodamaea_sp.']
Number of non-strain-specific duplicate species: 14


In [328]:
# Note to self: Changed "FULL_FINAL" fastas to "base_reference" so I can use them if I decide to make species-only fastas.

In [331]:
full_final_CHCs_descs = []

for seq in full_final_CHCs:
    full_final_CHCs_descs.append(seq.description)

entries_not_in_CHC_fasta = [
    assembly_ID 
    for assembly_ID in species_table['assembly_fullID_updated'] 
    if not any(assembly_ID in desc for desc in full_final_CHCs_descs)
]

entries_not_in_CHC_fasta

['yHMPu5000035666_nakaseomyces_bacillisporus_160613',
 'yHMPu5000034976_dekkera_anomala_160519.haplomerger2',
 'yHMPu5000037927_candida_apicola_180604',
 'yHMPu5000040956_metschnikowia_borealis_201018',
 'yHMPu5000040940_metschnikowia_matae_201018',
 'yHMPu5000041818_magnusiomyces_tetrasperma_170307',
 'yHMPu5000035658_starmera_amethionina_160613.haplomerger2',
 'yHMPu5000035694_hanseniaspora_occidentalis_var_citrica_160519.haplomerger2',
 'yHMPu5000035289_candida_infanticola_170713',
 'yHMPu5000038361_candida_versatilis_170713',
 'yHMPu5000037923_ascoidea_tarda_180604',
 'yHMPu5000034598_Candida_dendrica_SPADES',
 'yHMPu5000026105_candida_digboiensis_210210_SPADES.haplomerger2',
 'yHMPu5000038388_candida_inconspicua_170713',
 'yHMPu5000034993_candida_metapsilosis_170307.haplomerger2',
 'yHMPu5000035329_candida_saitoana_180604.haplomerger2',
 'yHMPu5000041848_candida_musae_170713',
 'yHMPu5000035337_cyberlindnera_mississippiensis_190924',
 'yHMPu5000034753_dipodascopsis_uninucleata_var

In [353]:
final_CHCs_strain_level = list(SeqIO.parse(local_path + 'genome_analyses/final_CHCs_strain_level.fasta', 'fasta'))

final_CHCs_strain_level_descs = []

for seq in final_CHCs_strain_level:
    final_CHCs_strain_level_descs.append(seq.description.lower())

entries_not_in_CHC_fasta = [
    name
    for name in species_strain_clade_df['species']
    if not any(name.lower() in desc for desc in final_CHCs_strain_level_descs)
]

entries_not_in_CHC_fasta

['Alloascoidea_hylecoeti_NRRL_Y-17703',
 'Alloascoidea_hylecoeti_JCM_7604',
 'Blastobotrys_adeninivorans_NRRL_Y-17692',
 'Blastobotrys_adeninivorans_LS3',
 'Geotrichum_candidum_CLIB_918',
 'Nadsonia_fulvescens_var._elongata',
 'Nadsonia_fulvescens_var._fulvescens',
 'Starmerella_apicola_NRRL_Y-2481',
 'Starmerella_apicola_NRRL_Y-50540',
 'Wickerhamiella_infanticola_NRRL_Y-17858',
 'Wickerhamiella_infanticola_DS02',
 'Wickerhamiella_versatilis_NRRL_Y-6652',
 'Wickerhamiella_versatilis_JCM_5958',
 'Yarrowia_lipolytica_NRRL_YB-423',
 'Yarrowia_lipolytica_CLIB_122',
 'Dipodascopsis_uninucleata_var._uninucleata',
 'Dipodascopsis_uninucleata_var._wickerhamii',
 'Lipomyces_sp._NRRL_Y-11553',
 'Candida_sp._NRRL_YB-4088',
 'Wickerhamomyces_sp._UFMG-CM-Y6624',
 'Wickerhamomyces_sp._NRRL_YB-2243',
 'Brettanomyces_anomalus_NRRL_Y-17522',
 'Brettanomyces_anomalus_YV396',
 'Candida_sp._NRRL_Y-12764',
 'Komagataella_pastoris_NRRL_Y-1603',
 'Komagataella_pastoris_ATCC_28485',
 'Komagataella_phaffii_NR

In [354]:
final_CLCs_strain_level = list(SeqIO.parse(local_path + 'genome_analyses/final_CLCs_strain_level.fasta', 'fasta'))

final_CLCs_strain_level_descs = []

for seq in final_CLCs_strain_level:
    final_CLCs_strain_level_descs.append(seq.description.lower())

entries_not_in_CLC_fasta = [
    name
    for name in species_strain_clade_df['species']
    if not any(name.lower() in desc for desc in final_CLCs_strain_level_descs)
]

entries_not_in_CLC_fasta

['Dipodascopsis_uninucleata_var._wickerhamii',
 'Candida_cabralensis',
 'Grigorovia_humatica',
 'Kluyveromyces_lactis_var._drosophilarum_NRRL_Y-8278',
 'Kluyveromyces_lactis_var._lactis_NRRL_Y-8279',
 'Lachancea_cidri_NRRL_Y-12634',
 'Hanseniaspora_occidentalis_var._citrica',
 'Candida_albicans_NRRL_Y-12983',
 'Candida_albicans_SC5314',
 'Candida_parapsilosis_NRRL_Y-12969',
 'Candida_sojae_GF41',
 'Metschnikowia_bicuspidata_var._californica',
 'Metschnikowia_bicuspidata_var._chathamia',
 'Metschnikowia_matae_var._matae_NRRL_Y-63736',
 'Schwanniomyces_occidentalis_var._persoonii',
 'Schwanniomyces_polymorphus_var._africanus',
 'Schwanniomyces_vanrijiae_var._yarrowii',
 'Teunomyces_stri',
 'Ascobotryozyma_cognata']

In [375]:
print('CLC outliers:')
for record in final_CLCs_strain_level:
    if len(record.seq) < 170 and '|clade=Outgroup' not in record.description:
        print(len(record.seq), record.description)
    elif len(record.seq) > 300 and '|clade=Outgroup' not in record.description:
        print(len(record.seq), record.description)

CLC outliers:
157 g002288.m1|ashbya_aceri.final|updated=Ashbya_aceri|clade=Saccharomycetales
149 g001773.m1|eremothecium_coryli.final|updated=Eremothecium_coryli|clade=Saccharomycetales
155 g004446.m1|eremothecium_cymbalariae.final|updated=Eremothecium_cymbalariae|clade=Saccharomycetales
157 g004603.m1|eremothecium_gossypii.final|updated=Eremothecium_gossypii|clade=Saccharomycetales
146 g003643.m1|eremothecium_sinecaudum.final|updated=Eremothecium_sinecaudum|clade=Saccharomycetales
134 g003313.m1|pichia_kudriavzevii.final|updated=Pichia_kudriavzevii_SD108|clade=Pichiales
760 g002381.m1|yHAB166_kazachstania_yakushimaensis_160519.final|updated=Grigorovia_yakushimaensis|clade=Saccharomycetales
125 g003519.m1|yHDO565_zygosaccharomyces_gambellarensis_180604.final|updated=Zygosaccharomyces_gambellarensis|clade=Saccharomycetales
125 g002791.m1|yHDO572_zygosaccharomyces_mellis_180604.final|updated=Zygosaccharomyces_mellis|clade=Saccharomycetales
156 g001370.m1|yHDO579_hagleromyces_aurorensis_1

In [378]:
print('CHC outliers:')
for record in final_CHCs_strain_level:
    if len(record.seq) < 1600 and '|clade=Outgroup' not in record.description:
        print(len(record.seq), record.description)
    elif len(record.seq) > 1700 and '|clade=Outgroup' not in record.description:
        print(len(record.seq), record.description)

CHC outliers:
1286 Metschnikowia_colocasiae.concatenated|clade=Serinetales
1518 spathaspora_gorwiae.concatenated|clade=Serinetales
1720 g004039.m1|yHDO584_brettanomyces_mucatilis_190924.haplomerger2.final|updated=botryozyma_mucatilis|clade=Trigonopsidales
1400 candida_digboiensis.concatenated|clade=Dipodascales
1908 g003973.m1|yHMPu5000034950_citeromyces_hawaiiensis_170307.haplomerger2.final|clade=Pichiales
1575 g003699.m1|yHMPu5000037231_lipomyces_yamanashiensis_210210.final|clade=Lipomycetales
1706 g004406.m1|yHMPu5000037921_ascoidea_asiatica_201018.final|clade=Ascoideales
1719 g005295.m1|yHMPu5000037922_ascoidea_rubescens_170210.final|clade=Ascoideales
1705 ascoidea_tarda.concatenated|clade=Ascoideales
1562 candida_inconspicua.concatenated|clade=Pichiales
1563 candida_insectamans.concatenated|updated=hemisphaericaspora_insectamans|clade=Serinetales
1578 Candida_jeffriesii.concatenated|updated=spathaspora_jeffriesii|clade=Serinetales
1829 g001551.m1|yHMPu5000041794_saccharomycopsis_l

In [399]:
final_CHCs_strain_level = list(SeqIO.parse(local_path + 'genome_analyses/final_CHCs_strain_level.fasta', 'fasta'))

pichiales_df = species_strain_clade_df[species_strain_clade_df['clade_name'] == 'Pichiales']

final_CHCs_strain_level_descs = []

for seq in final_CHCs_strain_level:
    if '|clade=Pichiales' in seq.description:
        final_CHCs_strain_level_descs.append(seq.description.lower())

entries_not_in_CHC_fasta = [
    name
    for name in pichiales_df['species']
    if not any(name.lower() in desc for desc in final_CHCs_strain_level_descs)
]

entries_not_in_CHC_fasta

['Brettanomyces_anomalus_NRRL_Y-17522',
 'Brettanomyces_anomalus_YV396',
 'Candida_sp._NRRL_Y-12764',
 'Komagataella_pastoris_NRRL_Y-1603',
 'Komagataella_pastoris_ATCC_28485',
 'Komagataella_phaffii_NRRL_Y-7556',
 'Komagataella_phaffii_GS115',
 'Ogataea_methylivora_NRRL_Y-17250',
 'Ogataea_methylovora_NRRL_Y-11996',
 'Ogataea_parapolymorpha_NRRL_YB-1982',
 'Ogataea_parapolymorpha_DL-1',
 'Ogataea_polymorpha_NRRL_Y-5445',
 'Ogataea_polymorpha_NCYC_495',
 'Pichia_kudriavzevii_NRRL_Y-5396',
 'Pichia_kudriavzevii_SD108']

In [407]:
final_CHCs_strain_level = list(SeqIO.parse(local_path + 'genome_analyses/final_CHCs_strain_level.fasta', 'fasta'))

serinetales_df = species_table[species_table['Order'] == 'Serinales']

final_CHCs_strain_level_descs = []

for seq in final_CHCs_strain_level:
    if '|clade=Serinetales' in seq.description: 
        parts = seq.description.split('|')
        
        if len(parts) > 2:
            curated_desc = parts[1]
        else:
            curated_desc = parts[0]

        curated_desc = curated_desc.replace('.final', '').replace('.concatenated', '').strip()
        
        final_CHCs_strain_level_descs.append(curated_desc)

descs_not_in_serinetales_df = [
    desc
    for desc in final_CHCs_strain_level_descs
    if not any(desc == assembly_ID for assembly_ID in serinetales_df['assembly_fullID_updated'])
]

descs_not_in_serinetales_df

['Metschnikowia_colocasiae',
 'Metschnikowia_hawaiiensis_NRRL_Y-17272',
 'Meyerozyma_guilliermondii_NRRL_Y-2075',
 'candida_metapsilosis',
 'candida_saitoana',
 'yHMPu5000035708_candida_flosculorum_160613.haplomerger2',
 'metschnikowia_fructicola',
 'updated=kodamaea_lidongshanica',
 'updated=hemisphaericaspora_insectamans',
 'metschnikowia_borealis',
 'updated=spathaspora_jeffriesii',
 'metschnikowia_pulcherrima',
 'updated=metschnikowia_ziziphicola',
 'metschnikowia_chrysoperlae',
 'metschnikowia_leonuri',
 'metschnikowia_shanxiensis']

In [414]:
final_CLCs_strain_level = list(SeqIO.parse(local_path + 'genome_analyses/final_CLCs_strain_level.fasta', 'fasta'))

print('CLC outliers:')
for record in final_CLCs_strain_level:
    if len(record.seq) < 170 and '|clade=Outgroup' not in record.description:
        print(len(record.seq), record.description)
    elif len(record.seq) > 300 and '|clade=Outgroup' not in record.description:
        print(len(record.seq), record.description)

# All major CLC outliers handled -- the rest are partial fragments containing the necessary interface residues.

CLC outliers:
157 g002288.m1|ashbya_aceri.final|updated=Ashbya_aceri|clade=Saccharomycetales
149 g001773.m1|eremothecium_coryli.final|updated=Eremothecium_coryli|clade=Saccharomycetales
155 g004446.m1|eremothecium_cymbalariae.final|updated=Eremothecium_cymbalariae|clade=Saccharomycetales
157 g004603.m1|eremothecium_gossypii.final|updated=Eremothecium_gossypii|clade=Saccharomycetales
146 g003643.m1|eremothecium_sinecaudum.final|updated=Eremothecium_sinecaudum|clade=Saccharomycetales
134 g003313.m1|pichia_kudriavzevii.final|updated=Pichia_kudriavzevii_SD108|clade=Pichiales
125 g003519.m1|yHDO565_zygosaccharomyces_gambellarensis_180604.final|updated=Zygosaccharomyces_gambellarensis|clade=Saccharomycetales
125 g002791.m1|yHDO572_zygosaccharomyces_mellis_180604.final|updated=Zygosaccharomyces_mellis|clade=Saccharomycetales
156 g001370.m1|yHDO579_hagleromyces_aurorensis_180604.final|updated=Hagleromyces_aurorensis|clade=Saccharomycetales
125 g001519.m1|yHDO603_zygosaccharomyces_sapae_190924.

In [415]:
final_CHCs_strain_level = list(SeqIO.parse(local_path + 'genome_analyses/final_CHCs_strain_level.fasta', 'fasta'))

print('CHC outliers:')
for record in final_CHCs_strain_level:
    if len(record.seq) < 1600 and '|clade=Outgroup' not in record.description:
        print(len(record.seq), record.description)
    elif len(record.seq) > 1700 and '|clade=Outgroup' not in record.description:
        print(len(record.seq), record.description)

# All major CHC outliers handled -- the rest are partial fragments containing the necessary interface residues. 

CHC outliers:
1286 Metschnikowia_colocasiae.concatenated|clade=Serinetales
1518 spathaspora_gorwiae.concatenated|clade=Serinetales
1720 g004039.m1|yHDO584_brettanomyces_mucatilis_190924.haplomerger2.final|updated=botryozyma_mucatilis|clade=Trigonopsidales
1400 candida_digboiensis.concatenated|clade=Dipodascales
1592 clavispora_lusitaniae.concatenated|clade=Serinetales
1575 g003699.m1|yHMPu5000037231_lipomyces_yamanashiensis_210210.final|clade=Lipomycetales
1706 g004406.m1|yHMPu5000037921_ascoidea_asiatica_201018.final|clade=Ascoideales
1719 g005295.m1|yHMPu5000037922_ascoidea_rubescens_170210.final|clade=Ascoideales
1705 ascoidea_tarda.concatenated|clade=Ascoideales
1562 candida_inconspicua.concatenated|clade=Pichiales
1563 candida_insectamans.concatenated|updated=hemisphaericaspora_insectamans|clade=Serinetales
1578 Candida_jeffriesii.concatenated|updated=spathaspora_jeffriesii|clade=Serinetales
1505 Metschnikowia_zizyphicola.concatenated|updated=metschnikowia_ziziphicola|clade=Serine

In [417]:
print('CLC outliers:')
for record in final_CLCs_strain_level:
    if len(record.seq) < 170 and '|clade=Outgroup' not in record.description:
        print(len(record.seq), record.description)
    elif len(record.seq) > 250 and '|clade=Outgroup' not in record.description:
        print(len(record.seq), record.description)

CLC outliers:
157 g002288.m1|ashbya_aceri.final|updated=Ashbya_aceri|clade=Saccharomycetales
149 g001773.m1|eremothecium_coryli.final|updated=Eremothecium_coryli|clade=Saccharomycetales
155 g004446.m1|eremothecium_cymbalariae.final|updated=Eremothecium_cymbalariae|clade=Saccharomycetales
157 g004603.m1|eremothecium_gossypii.final|updated=Eremothecium_gossypii|clade=Saccharomycetales
146 g003643.m1|eremothecium_sinecaudum.final|updated=Eremothecium_sinecaudum|clade=Saccharomycetales
134 g003313.m1|pichia_kudriavzevii.final|updated=Pichia_kudriavzevii_SD108|clade=Pichiales
125 g003519.m1|yHDO565_zygosaccharomyces_gambellarensis_180604.final|updated=Zygosaccharomyces_gambellarensis|clade=Saccharomycetales
125 g002791.m1|yHDO572_zygosaccharomyces_mellis_180604.final|updated=Zygosaccharomyces_mellis|clade=Saccharomycetales
156 g001370.m1|yHDO579_hagleromyces_aurorensis_180604.final|updated=Hagleromyces_aurorensis|clade=Saccharomycetales
125 g001519.m1|yHDO603_zygosaccharomyces_sapae_190924.

In [424]:
species_strain_clade_df

,species,clade_color,clade_name
0,Nakazawaea_anatomiae,#6A3906,Alaninetales
1,Nakazawaea_ernobii,#6A3906,Alaninetales
2,Nakazawaea_holstii,#6A3906,Alaninetales
3,Nakazawaea_ishiwadae,#6A3906,Alaninetales
4,Nakazawaea_laoshanensis,#6A3906,Alaninetales
...,...,...,...
1170,Tortispora_starmeri,#FF0090,Trigonopsidales
1171,Trigonopsis_californica,#FF0090,Trigonopsidales
1172,Trigonopsis_cantarellii,#FF0090,Trigonopsidales
1173,Trigonopsis_variabilis,#FF0090,Trigonopsidales


In [451]:
# Ensure NEW_Tip_ID is exactly the same as clade_name (excluding outgroup species)
from collections import Counter

Counter(species_table['NEW_Tip_ID']) == Counter(species_strain_clade_df.dropna(subset=['clade_name'])['species'])

True

In [475]:
# Check if both DFs contain the exact same pairs of strain + clade in the exact same counts

from collections import Counter

species_table_pairs = list(zip(species_table['NEW_Tip_ID'], species_table['Order']))

clade_mappings = {
    'Serinetales': 'Serinales',
    'Alaninetales': 'Alaninales',
    'Alloscoideatales': 'Alloascoideales'
}

clean_clade = species_strain_clade_df.dropna(subset=['clade_name'])
species_clade_df_pairs = list(zip(clean_clade['species'], clean_clade['clade_name'].replace(clade_mappings)))

is_exact_pair_match = Counter(species_table_pairs) == Counter(species_clade_df_pairs)

print(is_exact_pair_match)

# This being True confirms fully that I can use the species_table as my sole reference for my final confirmation below.

True


In [476]:
# Ensure no duplicates
print(species_table['NEW_Tip_ID'].duplicated().any())
print(species_strain_clade_df['species'].duplicated().any())

False
False


In [502]:
# This code:
# 1. Ensures each strain only appears once in the final CHC file
# 2. Ensures each strain is assigned the right clade in the final CHC file

final_CHCs_strain_level = list(SeqIO.parse(local_path + 'genome_analyses/final_CHCs_strain_level.fasta', 'fasta'))

strain_counts = Counter()

strains_w_clade_mismatch = []

adjusted_clade_mappings = {
    'Serinales': 'Serinetales',
    'Alaninales': 'Alaninetales',
    'Alloascoideales': 'Alloscoideatales'
}

for record in final_CHCs_strain_level:
    if 'HUMAN' in record.description:
        continue

    if '|updated=' in record.description:
        strain_name = record.description.split('|updated=')[1].split('|clade=')[0]
        
        if strain_name.lower() not in species_table['NEW_Tip_ID'].str.lower().values:
            print(f'{strain_name} (desc: {record.description}) has mismatch situation')
        else:
            order = species_table.loc[species_table['NEW_Tip_ID'].str.lower() == strain_name.lower(), 'Order'].iloc[0]
                
            for new_order_name, old_order_name in adjusted_clade_mappings.items():
                if new_order_name == order:
                    order = old_order_name
                    
            if order.lower() != record.description.split('|clade=')[1].lower():
                strains_w_clade_mismatch.append(record.description)

    elif '.concatenated' in record.description:
        strain_name = record.description.split('.concatenated')[0]

        if strain_name.lower() not in species_table['NEW_Tip_ID'].str.lower().values:
            print(f'{strain_name} (desc: {record.description}) has mismatch situation')
        else:
            order = species_table.loc[species_table['NEW_Tip_ID'].str.lower() == strain_name.lower(), 'Order'].iloc[0]
                
            for new_order_name, old_order_name in adjusted_clade_mappings.items():
                if new_order_name == order:
                    order = old_order_name
                    
            if order.lower() != record.description.split('|clade=')[1].lower():
                strains_w_clade_mismatch.append(record.description)

    else:
        strain_name = record.description
        if len(record.description.split('|')) > 2:
            strain_name_parts = record.description.split('|')[1].replace('.final', '').split('_')

            if 'yH' in strain_name_parts[0] and len(strain_name_parts) == 4:
                strain_name = f'{strain_name_parts[1]}_{strain_name_parts[2]}'
            else:
                strain_name = '_'.join(strain_name_parts)

            if strain_name.lower() not in species_table['NEW_Tip_ID'].str.lower().values:
                print(f'{strain_name} (desc: {record.description}) has mismatch situation')
            else:
                order = species_table.loc[species_table['NEW_Tip_ID'].str.lower() == strain_name.lower(), 'Order'].iloc[0]
                
                for new_order_name, old_order_name in adjusted_clade_mappings.items():
                    if new_order_name == order:
                        order = old_order_name
                    
                if order.lower() != record.description.split('|clade=')[1].lower():
                    strains_w_clade_mismatch.append(record.description)
            
        else:
            print(f'Handle {strain_name} manually')

    strain_counts[strain_name.lower()] += 1

print('-----------------------------------')
print('Potential duplicate strain entries:')
for strain, count in strain_counts.items():
    if count > 1:
        print(f'{strain}: {count}')

print('-----------------------------------')
print('Potential clade misassignments:')
strains_w_clade_mismatch

# All "mismatch situations" were checked manually and none were out of order; "mismatches" were simply due to the script's limitations.
# All potential duplicates were checked and found to not be duplicates.

alloascoidea_hylecoeti (desc: g007151.m1|alloascoidea_hylecoeti.final|clade=Alloscoideatales) has mismatch situation
blastobotrys_adeninivorans (desc: g002281.m1|arxula_adeninivorans.final|updated=blastobotrys_adeninivorans|clade=Dipodascales) has mismatch situation
brettanomyces_anomalus (desc: g000728.m1|brettanomyces_anomalus.final|clade=Pichiales) has mismatch situation
brettanomyces_anomalus (desc: g004611.m1|yHMPu5000034976_dekkera_anomala_160519.haplomerger2.final|updated=brettanomyces_anomalus|clade=Pichiales) has mismatch situation
starmerella_apicola (desc: g000288.m1|candida_apicola.final|updated=starmerella_apicola|clade=Dipodascales) has mismatch situation
starmerella_apicola (desc: g001645.m1|yHMPu5000037927_candida_apicola_180604.final|updated=starmerella_apicola|clade=Dipodascales) has mismatch situation
candida_auris (desc: g002166.m1|candida_auris.final|clade=Serinetales) has mismatch situation
candida_dubliniensis (desc: g005327.m1|candida_dubliniensis.final|clade=Se

[]

In [503]:
# This code:
# 1. Ensures each strain only appears once in the final CLC file
# 2. Ensures each strain is assigned the right clade in the final CLC file

final_CLCs_strain_level = list(SeqIO.parse(local_path + 'genome_analyses/final_CLCs_strain_level.fasta', 'fasta'))

strain_counts = Counter()

strains_w_clade_mismatch = []

adjusted_clade_mappings = {
    'Serinales': 'Serinetales',
    'Alaninales': 'Alaninetales',
    'Alloascoideales': 'Alloscoideatales'
}

for record in final_CLCs_strain_level:
    if 'HUMAN' in record.description:
        continue

    if '|updated=' in record.description:
        strain_name = record.description.split('|updated=')[1].split('|clade=')[0]
        
        if strain_name.lower() not in species_table['NEW_Tip_ID'].str.lower().values:
            print(f'{strain_name} (desc: {record.description}) has mismatch situation')
        else:
            order = species_table.loc[species_table['NEW_Tip_ID'].str.lower() == strain_name.lower(), 'Order'].iloc[0]
                
            for new_order_name, old_order_name in adjusted_clade_mappings.items():
                if new_order_name == order:
                    order = old_order_name
                    
            if order.lower() != record.description.split('|clade=')[1].lower():
                strains_w_clade_mismatch.append(record.description)

    elif '.concatenated' in record.description:
        strain_name = record.description.split('.concatenated')[0]

        if strain_name.lower() not in species_table['NEW_Tip_ID'].str.lower().values:
            print(f'{strain_name} (desc: {record.description}) has mismatch situation')
        else:
            order = species_table.loc[species_table['NEW_Tip_ID'].str.lower() == strain_name.lower(), 'Order'].iloc[0]
                
            for new_order_name, old_order_name in adjusted_clade_mappings.items():
                if new_order_name == order:
                    order = old_order_name
                    
            if order.lower() != record.description.split('|clade=')[1].lower():
                strains_w_clade_mismatch.append(record.description)

    else:
        strain_name = record.description
        if len(record.description.split('|')) > 2:
            strain_name_parts = record.description.split('|')[1].replace('.final', '').split('_')

            if 'yH' in strain_name_parts[0] and len(strain_name_parts) == 4:
                strain_name = f'{strain_name_parts[1]}_{strain_name_parts[2]}'
            else:
                strain_name = '_'.join(strain_name_parts)

            if strain_name.lower() not in species_table['NEW_Tip_ID'].str.lower().values:
                print(f'{strain_name} (desc: {record.description}) has mismatch situation')
            else:
                order = species_table.loc[species_table['NEW_Tip_ID'].str.lower() == strain_name.lower(), 'Order'].iloc[0]
                
                for new_order_name, old_order_name in adjusted_clade_mappings.items():
                    if new_order_name == order:
                        order = old_order_name
                    
                if order.lower() != record.description.split('|clade=')[1].lower():
                    strains_w_clade_mismatch.append(record.description)
            
        else:
            print(f'Handle {strain_name} manually')

    strain_counts[strain_name.lower()] += 1

print('-----------------------------------')
print('Potential duplicate strain entries:')
for strain, count in strain_counts.items():
    if count > 1:
        print(f'{strain}: {count}')

print('-----------------------------------')
print('Potential clade misassignments:')
strains_w_clade_mismatch

# All "mismatch situations" were checked manually and none were out of order; "mismatches" were simply due to the script's limitations.
# All potential duplicates were checked and found to not be duplicates.

candida_albicans (desc: g003204.m1|candida_albicans.final|clade=Serinetales) has mismatch situation
candida_albicans (desc: g005237.m1|yHMPu5000035004_candida_albicans_190924.final|clade=Serinetales) has mismatch situation
candida_parapsilosis (desc: g000857.m1|yHMPu5000034995_candida_parapsilosis_170307.final|clade=Serinetales) has mismatch situation
yHMPu5000034711_kluyveromyces_lactis_var_drosophilarum_160519 (desc: g004243.m1|yHMPu5000034711_kluyveromyces_lactis_var_drosophilarum_160519.final|clade=Saccharomycetales) has mismatch situation
yHMPu5000034712_kluyveromyces_lactis_var_lactis_160519 (desc: g000565.m1|yHMPu5000034712_kluyveromyces_lactis_var_lactis_160519.final|clade=Saccharomycetales) has mismatch situation
yHMPu5000041781_schwanniomyces_occidentalis_var_persoonii_190924 (desc: g001913.m1|yHMPu5000041781_schwanniomyces_occidentalis_var_persoonii_190924.final|clade=Serinetales) has mismatch situation
yHMPu5000037868_metschnikowia_bicuspidata_var_californica_180604.haplome

[]

In [504]:
# TODO:
# Assign outgroup clade --> DONE
# Assign clades to the 6 entries whose names were not updated automatically --> DONE
# Get rid of duplicate entries in the CLC file --> DONE
# Handle outliers in both the CHC and CLC files --> DONE
# Generate preliminary graphs for Connor --> DONE
# Finalize strain-specific distinctions (aka get the number of entries in both files 1154) --> DONE
# Generate new graphs and redo formatting --> DONE
# Do final check to confirm no remaining duplicate entries in both the CHC and CLC files **IMPORTANT SO 1154 NUMBER IS TRUE** --> DONE
# Do final check where you confirm for both CHC and CLC that all seqs are assigned the right clade --> DONE

In [ ]:
# IMPORTANT: REVIEW ALL OF THIS CODE BELOW BEFORE FINALIZING THE GRAPHS

In [ ]:
# ADD HUMAN CHC17 AND HUMAN CLCa (Isoform Non-brain) TO THE TOP BEFORE RUNNING THE CODE BELOW

In [516]:
import subprocess

with open(local_path + 'genome_analyses/final_CLCs_strain_level.fasta', 'r') as handle: # Re-check this before running
    seqs_str = handle.read()

# This was used for the 332 species dataset --localpair makes this take way too long:
#cmd = [
#    'mafft',
#    '--localpair',
#    '--maxiterate', '1000',
#    '--thread', '10',
#    '--quiet',
#    '-'
#]

# Use this for quick testing:
#cmd = [
#    'mafft',
#    '--retree', '2',    # FFT-NS-2 strategy
#    '--maxiterate', '0', # No iterations (makes it fast)
#    '--thread', '10',
#    '--quiet',
#    '-'
#]

# Use the FFT-NS-i strategy below for the final graphs (best accuracy and prevents extremely long run / running out of memory)
cmd = [
    'mafft',
    '--maxiterate', '1000',
    '--thread', '10',
    '--quiet',
    '-'
]

process = subprocess.run(
    cmd,
    input=seqs_str,
    text=True,
    capture_output=True
)

if process.returncode != 0:
    print('MAFFT error:', process.stderr)
else:
    aligned_fasta = process.stdout
    print('Alignment completed successfully')
    with open(local_path + 'genome_analyses/final_CLCs_strain_level_aligned.fasta', 'w') as out_f: # Re-check this before running
        out_f.write(aligned_fasta)

Alignment completed successfully


In [517]:
from collections import defaultdict

clade_groups = defaultdict(list)

with open(local_path + 'genome_analyses/final_CLCs_strain_level_aligned.fasta') as handle: # Re-check this before running
    for seq_record in SeqIO.parse(handle, 'fasta'):
        if 'clade=' not in seq_record.description:
            print(f'Clade assignment not found in {seq_record.description}.')
            continue
            
        clade_name = seq_record.description.split('clade=')[1].strip()
        clade_groups[clade_name].append(seq_record)

for clade, seqs in clade_groups.items():
    clade = clade.replace('/', '_').replace(' ', '-')
    with open(local_path + f'1k-species-by-clade/CLC1-fastas-by-clade/{clade}_aligned_CLCs.fasta', 'w') as out_f: # Re-check before running
        SeqIO.write(seqs, out_f, 'fasta')

Clade assignment not found in sp|P09496-2|CLCA_HUMAN Isoform Non-brain of Clathrin light chain A OS=Homo sapiens OX=9606 GN=CLTA.
